# Explainable Multilingual Legal Meaning Preservation (JUDGEBERT extension)
**CSET 346 - NLP project.** A JUDGEBERT-style cross-encoder extended to **English + Hindi**, two legal domains
(**insurance**, **tenancy**), with **explainable** outputs and the five required experiments.

For each original-simplified pair the system outputs:
**LMP score (1-10)**, **decision** (Preserved / Partially / Not Preserved), **legal error types** and a **short explanation**.

| Research gap (JUDGEBERT paper) | Where it is addressed |
|---|---|
| French-only | English + Hindi, multilingual encoder, cross-lingual zero-shot (Exp. 2) |
| Single legal domain, no OOD test | insurance <-> tenancy transfer (Exp. 3a) |
| Sentence-level only | sentence-only vs. contextual input (Exp. 3b) |
| Score only, no explanation | multi-task heads + Integrated Gradients + rule evidence (Exp. 5) |
| Single simplification system, no fine-grained robustness test | controlled legal-edit suite (Exp. 5a), multi-LLM data generation (Sec. 3) |

**Before running:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.

> **Important:** if you do not upload a human-annotated CSV, the notebook runs on *silver* data
> (gold scores produced by rules). That checks the full pipeline, but those numbers are **not** paper
> results. Your paper's results must come from your annotated 300+ pairs (Section 3).

## 1. Setup

In [ ]:
!pip -q install sentence-transformers sacrebleu krippendorff tabulate gradio
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to T4 GPU")

## 2. Project code
Each cell below writes one module of the `lmp` package (identical to the GitHub repo).
If you already pushed the repo, you can instead run `!git clone https://github.com/<you>/judgebert-lmp && %cd judgebert-lmp` and skip these cells.

In [ ]:
import os
os.makedirs('lmp', exist_ok=True)
os.makedirs('data', exist_ok=True)

In [ ]:
%%writefile lmp/__init__.py
"""Explainable Multilingual Legal Meaning Preservation (a JUDGEBERT extension to English + Hindi)."""
__version__ = "1.0.0"


In [ ]:
%%writefile lmp/config.py
"""Central configuration for the Explainable Multilingual LMP project."""
from dataclasses import dataclass, field
from typing import List, Optional

# Legal error types (multi-label). Matches the course brief: omission, hallucination,
# contradiction, obligation/permission, conditions, exceptions, scope, dates/numbers, entities.
ERROR_TYPES: List[str] = [
    "omission",
    "hallucination",
    "contradiction",
    "obligation_permission",
    "condition",
    "exception",
    "scope",
    "number_date",
    "entity",
]

DECISIONS: List[str] = ["Not Preserved", "Partially Preserved", "Preserved"]


@dataclass
class Config:
    # ---- model ----
    # Good multilingual choices that cover English + Hindi:
    #   "xlm-roberta-base"            (default, robust)
    #   "google/muril-base-cased"     (trained on 17 Indian languages + English)
    #   "ai4bharat/IndicBERTv2-MLM-only"
    #   "xlm-roberta-large"           (best quality, needs A100 / small batch)
    model_name: str = "xlm-roberta-base"
    max_len: int = 256
    dropout: float = 0.1

    # ---- training ----
    epochs: int = 12
    batch_size: int = 16
    lr: float = 2e-5
    head_lr: float = 1e-3
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    patience: int = 4
    min_epochs: int = 3          # no checkpoint selection / early stopping before this epoch
    seeds: List[int] = field(default_factory=lambda: [42, 43, 44])
    fp16: bool = True

    # ---- loss ----
    # Over-estimating legal meaning preservation is the dangerous direction
    # (it approves bad simplifications), so it is penalised more heavily.
    over_penalty: float = 2.0
    w_decision: float = 0.5
    w_errors: float = 0.5

    # ---- data ----
    use_context: bool = False
    data_augmentation: bool = True   # add identical (=10) and unrelated (=1) pairs to training
    test_size: float = 0.2
    val_size: float = 0.1

    # ---- decision thresholds on the 1-10 LMP scale (paper: 7-10 accurate, 2-6 imprecise, 1 off-track) ----
    preserved_min: float = 6.5
    not_preserved_max: float = 2.0

    # ---- baselines ----
    bertscore_model: str = "xlm-roberta-base"
    bertscore_layer: int = 9
    sbert_models: List[str] = field(default_factory=lambda: [
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        "sentence-transformers/LaBSE",
    ])
    run_neural_baselines: bool = True

    output_dir: str = "outputs"
    device: Optional[str] = None


def score_to_decision(score: float, cfg: Config) -> str:
    if score >= cfg.preserved_min:
        return DECISIONS[2]
    if score <= cfg.not_preserved_max:
        return DECISIONS[0]
    return DECISIONS[1]


def score_to_decision_id(score: float, cfg: Config) -> int:
    return DECISIONS.index(score_to_decision(score, cfg))


In [ ]:
%%writefile lmp/seed_data.py
"""Parallel English-Hindi seed corpus of legal clauses (insurance + tenancy).

Each concept has, for both languages:
    context  - the definition / heading that precedes the clause in the document
    original - the clause in formal legal language
    simple   - a hand-written, meaning-preserving plain-language simplification

These clauses were written for this project in the style of Indian insurance policies
and residential rent agreements (they are not copied from any real document).

IMPORTANT: this seed corpus + the rule-based perturbations in `perturb.py` produce
*silver* LMP labels. They let you run and debug the whole pipeline today and are useful
as data augmentation, but the research results in your paper must come from the
human-annotated dataset (see `annotation_template.csv` and `data.load_annotated_csv`).
"""

SEED = [
    # ============================ INSURANCE ============================
    {
        "cid": "INS01", "domain": "insurance",
        "en": (
            "Definitions: 'Loss' means any physical loss of or damage to the insured property during the period of insurance.",
            "The Insured shall notify the Insurer in writing of any loss within 30 days of its occurrence, failing which the claim shall not be admissible.",
            "The insured must tell the insurer in writing about any loss within 30 days. Otherwise, the claim will not be accepted.",
        ),
        "hi": (
            "परिभाषाएँ: 'हानि' का अर्थ बीमा अवधि के दौरान बीमित संपत्ति की कोई भी भौतिक हानि या क्षति है।",
            "बीमाधारक किसी भी हानि के घटित होने के 30 दिनों के भीतर बीमाकर्ता को लिखित रूप में सूचित करेगा, अन्यथा दावा स्वीकार्य नहीं होगा।",
            "बीमाधारक को किसी भी नुकसान की जानकारी 30 दिनों के अंदर बीमाकर्ता को लिखकर देनी होगी। ऐसा न करने पर दावा स्वीकार नहीं होगा।",
        ),
    },
    {
        "cid": "INS02", "domain": "insurance",
        "en": (
            "Section II lists the optional extensions available under this Policy, including the Flood Extension.",
            "The Insurer shall not be liable for any loss or damage caused by flood, unless the Flood Extension has been purchased and the additional premium paid.",
            "The insurer will not pay for flood damage unless you bought the Flood Extension and paid the extra premium.",
        ),
        "hi": (
            "खंड II में इस पॉलिसी के अंतर्गत उपलब्ध वैकल्पिक विस्तार सूचीबद्ध हैं, जिनमें बाढ़ विस्तार शामिल है।",
            "बीमाकर्ता बाढ़ से हुई किसी भी हानि या क्षति के लिए उत्तरदायी नहीं होगा, सिवाय इसके कि बाढ़ विस्तार कवर खरीदा गया हो और अतिरिक्त प्रीमियम का भुगतान किया गया हो।",
            "बीमाकर्ता बाढ़ से हुए नुकसान का भुगतान नहीं करेगा, सिवाय इसके कि आपने बाढ़ विस्तार कवर खरीदा हो और अतिरिक्त प्रीमियम चुकाया हो।",
        ),
    },
    {
        "cid": "INS03", "domain": "insurance",
        "en": (
            "'Replacement value' means the cost of replacing the property with new property of the same kind and capacity.",
            "Where the sum insured is less than 80 percent of the replacement value of the property, the indemnity payable shall be reduced proportionately.",
            "If the sum insured is less than 80% of the replacement value, the payment will be reduced proportionately.",
        ),
        "hi": (
            "'प्रतिस्थापन मूल्य' का अर्थ संपत्ति को उसी प्रकार और क्षमता की नई संपत्ति से बदलने की लागत है।",
            "जहाँ बीमित राशि संपत्ति के प्रतिस्थापन मूल्य के 80 प्रतिशत से कम है, वहाँ देय क्षतिपूर्ति आनुपातिक रूप से घटा दी जाएगी।",
            "यदि बीमित राशि प्रतिस्थापन मूल्य के 80% से कम है, तो भुगतान उसी अनुपात में कम कर दिया जाएगा।",
        ),
    },
    {
        "cid": "INS04", "domain": "insurance",
        "en": (
            "The period of insurance is the twelve-month period stated in the Schedule.",
            "The policyholder shall pay the premium in full prior to the commencement of the period of insurance, and no cover shall attach until such payment is received.",
            "The policyholder must pay the full premium before the policy starts. There is no cover until the payment is received.",
        ),
        "hi": (
            "बीमा अवधि अनुसूची में उल्लिखित बारह माह की अवधि है।",
            "पॉलिसीधारक बीमा अवधि के प्रारंभ से पूर्व प्रीमियम का पूर्ण भुगतान करेगा, और ऐसा भुगतान प्राप्त होने तक कोई कवर लागू नहीं होगा।",
            "पॉलिसीधारक को पॉलिसी शुरू होने से पहले पूरा प्रीमियम चुकाना होगा। भुगतान मिलने तक कोई कवर नहीं होगा।",
        ),
    },
    {
        "cid": "INS05", "domain": "insurance",
        "en": (
            "This Section covers loss of or damage to the insured property by burglary or theft.",
            "In the event of theft, the Insured must lodge a First Information Report with the police within 48 hours and furnish a copy thereof to the Insurer.",
            "If something is stolen, the insured must file an FIR with the police within 48 hours and give a copy to the insurer.",
        ),
        "hi": (
            "यह खंड सेंधमारी या चोरी द्वारा बीमित संपत्ति की हानि या क्षति को कवर करता है।",
            "चोरी की स्थिति में, बीमाधारक 48 घंटों के भीतर पुलिस में प्रथम सूचना रिपोर्ट दर्ज कराएगा और उसकी एक प्रति बीमाकर्ता को प्रस्तुत करेगा।",
            "यदि कुछ चोरी हो जाए, तो बीमाधारक को 48 घंटों के अंदर पुलिस में FIR दर्ज करानी होगी और उसकी कॉपी बीमाकर्ता को देनी होगी।",
        ),
    },
    {
        "cid": "INS06", "domain": "insurance",
        "en": (
            "General Conditions: the following conditions apply to all Sections of this Policy.",
            "The Insurer may cancel this Policy by giving 15 days' notice in writing to the Insured, in which event a pro-rata refund of premium shall be made.",
            "The insurer may cancel this policy by giving the insured 15 days' written notice. The insurer must then refund part of the premium.",
        ),
        "hi": (
            "सामान्य शर्तें: निम्नलिखित शर्तें इस पॉलिसी के सभी खंडों पर लागू होती हैं।",
            "बीमाकर्ता बीमाधारक को 15 दिनों का लिखित नोटिस देकर इस पॉलिसी को रद्द कर सकता है, जिस स्थिति में प्रीमियम की आनुपातिक वापसी की जाएगी।",
            "बीमाकर्ता बीमाधारक को 15 दिन का लिखित नोटिस देकर यह पॉलिसी रद्द कर सकता है। तब बीमाकर्ता को प्रीमियम का कुछ हिस्सा लौटाना होगा।",
        ),
    },
    {
        "cid": "INS07", "domain": "insurance",
        "en": (
            "Section I covers accidental loss of or damage to the insured property, subject to the exclusions below.",
            "No claim shall be payable in respect of any loss arising from wear and tear, gradual deterioration or mechanical breakdown.",
            "The policy does not pay for any loss caused by wear and tear, slow deterioration or mechanical breakdown.",
        ),
        "hi": (
            "खंड I नीचे दिए गए अपवादों के अधीन बीमित संपत्ति की आकस्मिक हानि या क्षति को कवर करता है।",
            "टूट-फूट, क्रमिक क्षय या यांत्रिक खराबी से उत्पन्न किसी भी हानि के संबंध में कोई दावा देय नहीं होगा।",
            "पॉलिसी टूट-फूट, धीरे-धीरे खराब होने या मशीनी खराबी से हुए किसी भी नुकसान का भुगतान नहीं करती है।",
        ),
    },
    {
        "cid": "INS08", "domain": "insurance",
        "en": (
            "General Conditions: the following conditions apply to all Sections of this Policy.",
            "The Insured shall take all reasonable steps to safeguard the insured property from loss or damage, and failure to do so may result in the claim being reduced.",
            "The insured must take all reasonable steps to protect the property. If not, the claim may be reduced.",
        ),
        "hi": (
            "सामान्य शर्तें: निम्नलिखित शर्तें इस पॉलिसी के सभी खंडों पर लागू होती हैं।",
            "बीमाधारक बीमित संपत्ति को हानि या क्षति से बचाने के लिए सभी उचित कदम उठाएगा, और ऐसा न करने पर दावे की राशि घटाई जा सकती है।",
            "बीमाधारक को संपत्ति की सुरक्षा के लिए सभी उचित कदम उठाने होंगे। ऐसा न करने पर दावा कम किया जा सकता है।",
        ),
    },
    {
        "cid": "INS09", "domain": "insurance",
        "en": (
            "'Sum Insured' means the amount stated in the Schedule against each Section.",
            "The maximum liability of the Insurer under this Policy shall not exceed the Sum Insured of Rs. 5,00,000 for any one occurrence.",
            "The insurer will pay at most Rs. 5,00,000 for any one event.",
        ),
        "hi": (
            "'बीमित राशि' का अर्थ अनुसूची में प्रत्येक खंड के सामने उल्लिखित राशि है।",
            "इस पॉलिसी के अंतर्गत बीमाकर्ता का अधिकतम दायित्व किसी एक घटना के लिए रु. 5,00,000 की बीमित राशि से अधिक नहीं होगा।",
            "बीमाकर्ता किसी एक घटना के लिए अधिकतम रु. 5,00,000 तक का भुगतान करेगा।",
        ),
    },
    {
        "cid": "INS10", "domain": "insurance",
        "en": (
            "Arbitration under this Policy is governed by the Arbitration and Conciliation Act.",
            "Any dispute regarding the quantum of loss shall be referred to arbitration, provided that the Insurer has admitted liability under this Policy.",
            "Any dispute about the amount of loss must go to arbitration, provided the insurer has accepted liability.",
        ),
        "hi": (
            "इस पॉलिसी के अंतर्गत मध्यस्थता माध्यस्थम् और सुलह अधिनियम द्वारा शासित है।",
            "हानि की मात्रा के संबंध में कोई भी विवाद मध्यस्थता को भेजा जाएगा, बशर्ते बीमाकर्ता ने इस पॉलिसी के अंतर्गत दायित्व स्वीकार किया हो।",
            "नुकसान की राशि पर कोई भी विवाद मध्यस्थता में जाएगा, बशर्ते बीमाकर्ता ने दायित्व स्वीकार किया हो।",
        ),
    },
    {
        "cid": "INS11", "domain": "insurance",
        "en": (
            "'Deductible' means the amount of each claim that the Insured must bear before the Insurer pays.",
            "The deductible of Rs. 10,000 shall be borne by the Insured in respect of each and every claim.",
            "The insured must pay the first Rs. 10,000 of each claim.",
        ),
        "hi": (
            "'कटौती राशि' का अर्थ प्रत्येक दावे की वह राशि है जो बीमाकर्ता के भुगतान से पहले बीमाधारक को वहन करनी होती है।",
            "प्रत्येक दावे के संबंध में रु. 10,000 की कटौती राशि बीमाधारक द्वारा वहन की जाएगी।",
            "हर दावे के पहले रु. 10,000 बीमाधारक को खुद चुकाने होंगे।",
        ),
    },
    {
        "cid": "INS12", "domain": "insurance",
        "en": (
            "'Insured vehicle' means the vehicle described in the Schedule of this Policy.",
            "This Policy shall automatically terminate upon the sale or transfer of the insured vehicle, unless the Insurer's prior written consent has been obtained.",
            "This policy ends automatically when the vehicle is sold or transferred, unless the insurer agreed in writing beforehand.",
        ),
        "hi": (
            "'बीमित वाहन' का अर्थ इस पॉलिसी की अनुसूची में वर्णित वाहन है।",
            "बीमित वाहन की बिक्री या हस्तांतरण पर यह पॉलिसी स्वतः समाप्त हो जाएगी, जब तक कि बीमाकर्ता की पूर्व लिखित सहमति प्राप्त न की गई हो।",
            "वाहन बेचने या हस्तांतरित करने पर यह पॉलिसी अपने आप खत्म हो जाएगी, जब तक कि बीमाकर्ता ने पहले लिखित में सहमति न दी हो।",
        ),
    },
    # ============================ TENANCY ============================
    {
        "cid": "TEN01", "domain": "tenancy",
        "en": (
            "This Rent Agreement is made between the Landlord and the Tenant for the premises described in Schedule A.",
            "The Tenant shall pay the monthly rent of Rs. 25,000 on or before the 5th day of each calendar month.",
            "The tenant must pay the monthly rent of Rs. 25,000 by the 5th of every month.",
        ),
        "hi": (
            "यह किराया अनुबंध अनुसूची क में वर्णित परिसर के लिए मकान-मालिक और किरायेदार के बीच किया गया है।",
            "किरायेदार प्रत्येक कैलेंडर माह की 5 तारीख को या उससे पूर्व रु. 25,000 का मासिक किराया अदा करेगा।",
            "किरायेदार को हर महीने की 5 तारीख तक रु. 25,000 मासिक किराया देना होगा।",
        ),
    },
    {
        "cid": "TEN02", "domain": "tenancy",
        "en": (
            "'Premises' means the residential flat described in Schedule A, including fittings and fixtures.",
            "The Tenant shall not sublet, assign or part with possession of the premises or any part thereof without the prior written consent of the Landlord.",
            "The tenant must not sublet or hand over any part of the premises without the landlord's written permission first.",
        ),
        "hi": (
            "'परिसर' का अर्थ अनुसूची क में वर्णित आवासीय फ्लैट है, जिसमें फिटिंग और फिक्स्चर शामिल हैं।",
            "किरायेदार मकान-मालिक की पूर्व लिखित सहमति के बिना परिसर या उसके किसी भाग को उप-किराये पर नहीं देगा, न ही उसका कब्ज़ा किसी अन्य को सौंपेगा।",
            "किरायेदार मकान-मालिक की लिखित अनुमति के बिना परिसर का कोई भी हिस्सा किसी और को किराये पर नहीं दे सकता।",
        ),
    },
    {
        "cid": "TEN03", "domain": "tenancy",
        "en": (
            "The Tenant has paid a security deposit to the Landlord at the time of signing this agreement.",
            "The security deposit of Rs. 1,00,000 shall be refunded by the Landlord within 30 days of the Tenant vacating the premises, after deducting any arrears of rent.",
            "The landlord must return the Rs. 1,00,000 security deposit within 30 days after the tenant leaves, minus any unpaid rent.",
        ),
        "hi": (
            "किरायेदार ने इस अनुबंध पर हस्ताक्षर के समय मकान-मालिक को प्रतिभूति राशि का भुगतान किया है।",
            "मकान-मालिक, किरायेदार द्वारा परिसर खाली किए जाने के 30 दिनों के भीतर, किराये के बकाया की कटौती के पश्चात रु. 1,00,000 की प्रतिभूति राशि वापस करेगा।",
            "किरायेदार के घर खाली करने के 30 दिनों के अंदर मकान-मालिक को रु. 1,00,000 की सिक्योरिटी राशि लौटानी होगी, बकाया किराया काटकर।",
        ),
    },
    {
        "cid": "TEN04", "domain": "tenancy",
        "en": (
            "Termination: this clause sets out how the agreement may be ended before its expiry.",
            "Either party may terminate this agreement by giving the other party not less than two months' notice in writing.",
            "Either side can end this agreement by giving the other at least two months' written notice.",
        ),
        "hi": (
            "समाप्ति: यह खंड बताता है कि अनुबंध को उसकी अवधि पूरी होने से पहले कैसे समाप्त किया जा सकता है।",
            "कोई भी पक्ष दूसरे पक्ष को कम से कम दो माह का लिखित नोटिस देकर इस अनुबंध को समाप्त कर सकता है।",
            "कोई भी पक्ष दूसरे पक्ष को कम से कम दो महीने का लिखित नोटिस देकर यह अनुबंध खत्म कर सकता है।",
        ),
    },
    {
        "cid": "TEN05", "domain": "tenancy",
        "en": (
            "The monthly rent payable under this agreement is stated in Clause 2.",
            "The rent shall be increased by 5 percent upon the expiry of every eleven months, unless otherwise agreed between the parties in writing.",
            "The rent will go up by 5% every eleven months, unless both sides agree otherwise in writing.",
        ),
        "hi": (
            "इस अनुबंध के अंतर्गत देय मासिक किराया खंड 2 में उल्लिखित है।",
            "प्रत्येक ग्यारह माह की समाप्ति पर किराये में 5 प्रतिशत की वृद्धि की जाएगी, जब तक कि पक्षकारों के बीच लिखित रूप में अन्यथा सहमति न हो।",
            "हर ग्यारह महीने बाद किराया 5% बढ़ेगा, जब तक कि दोनों पक्ष लिखित में कुछ और तय न करें।",
        ),
    },
    {
        "cid": "TEN06", "domain": "tenancy",
        "en": (
            "Maintenance and Repairs.",
            "The Landlord shall be responsible for all major structural repairs, whereas minor day-to-day repairs shall be carried out by the Tenant at his own cost.",
            "The landlord must handle all major structural repairs. The tenant must pay for small everyday repairs.",
        ),
        "hi": (
            "रखरखाव और मरम्मत।",
            "मकान-मालिक सभी बड़ी संरचनात्मक मरम्मत के लिए उत्तरदायी होगा, जबकि छोटी दैनिक मरम्मत किरायेदार द्वारा अपने व्यय पर कराई जाएगी।",
            "सभी बड़ी संरचनात्मक मरम्मत मकान-मालिक को करानी होगी। छोटी रोज़मर्रा की मरम्मत का खर्च किरायेदार को उठाना होगा।",
        ),
    },
    {
        "cid": "TEN07", "domain": "tenancy",
        "en": (
            "Landlord's Rights.",
            "The Landlord or his authorised agent may enter the premises for inspection upon giving the Tenant at least 24 hours' prior notice.",
            "The landlord or his agent can enter the premises for inspection if he gives the tenant at least 24 hours' notice.",
        ),
        "hi": (
            "मकान-मालिक के अधिकार।",
            "मकान-मालिक या उसका अधिकृत एजेंट किरायेदार को कम से कम 24 घंटे का पूर्व नोटिस देकर निरीक्षण हेतु परिसर में प्रवेश कर सकता है।",
            "अगर मकान-मालिक किरायेदार को कम से कम 24 घंटे पहले बता दे, तो वह या उसका एजेंट जाँच के लिए परिसर में आ सकता है।",
        ),
    },
    {
        "cid": "TEN08", "domain": "tenancy",
        "en": (
            "Use of Premises.",
            "The premises shall be used solely for residential purposes and the Tenant shall not carry on any commercial activity therein.",
            "The premises must be used only as a home. The tenant must not run any business there.",
        ),
        "hi": (
            "परिसर का उपयोग।",
            "परिसर का उपयोग केवल आवासीय प्रयोजनों हेतु किया जाएगा और किरायेदार उसमें कोई वाणिज्यिक गतिविधि नहीं करेगा।",
            "परिसर का उपयोग केवल रहने के लिए करना होगा। किरायेदार वहाँ कोई व्यापार नहीं कर सकता।",
        ),
    },
    {
        "cid": "TEN09", "domain": "tenancy",
        "en": (
            "Default and Remedies.",
            "In case of default in payment of rent for two consecutive months, the Landlord shall be entitled to terminate the tenancy forthwith.",
            "If the tenant does not pay rent for two months in a row, the landlord can end the tenancy immediately.",
        ),
        "hi": (
            "चूक और उपचार।",
            "लगातार दो माह तक किराये के भुगतान में चूक होने की स्थिति में, मकान-मालिक तत्काल किरायेदारी समाप्त करने का हकदार होगा।",
            "यदि किरायेदार लगातार दो महीने किराया नहीं देता, तो मकान-मालिक तुरंत किरायेदारी खत्म कर सकता है।",
        ),
    },
    {
        "cid": "TEN10", "domain": "tenancy",
        "en": (
            "Utilities and Other Charges.",
            "The Tenant shall bear the electricity and water charges as per actual consumption, as recorded by the separate meters installed in the premises.",
            "The tenant must pay the electricity and water bills based on actual use, as shown by the separate meters.",
        ),
        "hi": (
            "उपयोगिताएँ और अन्य शुल्क।",
            "किरायेदार परिसर में लगे अलग मीटरों द्वारा दर्ज वास्तविक खपत के अनुसार बिजली और पानी के शुल्क वहन करेगा।",
            "किरायेदार को अलग मीटरों में दर्ज असली खपत के हिसाब से बिजली और पानी का बिल चुकाना होगा।",
        ),
    },
    {
        "cid": "TEN11", "domain": "tenancy",
        "en": (
            "Term of Agreement.",
            "This agreement shall be for a period of eleven months commencing from 1st April 2026, and may be renewed by mutual consent of the parties.",
            "This agreement lasts eleven months starting 1 April 2026. It can be renewed if both sides agree.",
        ),
        "hi": (
            "अनुबंध की अवधि।",
            "यह अनुबंध 1 अप्रैल 2026 से प्रारंभ होकर ग्यारह माह की अवधि के लिए होगा, और पक्षकारों की आपसी सहमति से नवीनीकृत किया जा सकता है।",
            "यह अनुबंध 1 अप्रैल 2026 से ग्यारह महीने के लिए है। दोनों पक्ष सहमत हों तो इसे आगे बढ़ाया जा सकता है।",
        ),
    },
    {
        "cid": "TEN12", "domain": "tenancy",
        "en": (
            "Handing Over of Possession.",
            "The Tenant shall hand over vacant and peaceful possession of the premises to the Landlord upon expiry or earlier termination of this agreement, failing which he shall be liable to pay double the monthly rent as damages.",
            "When the agreement ends, the tenant must give the empty premises back to the landlord. If he does not, he must pay double the monthly rent as damages.",
        ),
        "hi": (
            "कब्ज़ा सौंपना।",
            "इस अनुबंध की समाप्ति या पूर्व समापन पर किरायेदार परिसर का खाली और शांतिपूर्ण कब्ज़ा मकान-मालिक को सौंपेगा, ऐसा न करने पर वह हर्जाने के रूप में मासिक किराये का दोगुना भुगतान करने का दायी होगा।",
            "अनुबंध खत्म होने पर किरायेदार को खाली घर मकान-मालिक को लौटाना होगा। ऐसा न करने पर उसे हर्जाने के रूप में दोगुना मासिक किराया देना होगा।",
        ),
    },
]


def seed_rows():
    """Flatten SEED into rows: one per (concept, language)."""
    rows = []
    for item in SEED:
        for lang in ("en", "hi"):
            ctx, orig, simp = item[lang]
            rows.append({
                "source_id": f"{item['cid']}-{lang}",
                "concept_id": item["cid"],
                "domain": item["domain"],
                "lang": lang,
                "context": ctx,
                "original": orig,
                "faithful": simp,
            })
    return rows


In [ ]:
%%writefile lmp/perturb.py
"""Controlled legal perturbations (English + Hindi) and silver-label generation.

Each perturbation takes a faithful simplification and introduces exactly one legally
meaningful change (obligation -> permission, dropped exception, changed number, ...).
They serve three purposes:
  1. silver training/augmentation data (`build_silver_dataset`)
  2. the robustness test suite for Experiment 5 (`perturbation_suite`)
  3. identical / unrelated sanity pairs for Experiment 4 (`build_sanity_pairs`)
"""
import random
import re
from typing import Callable, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

from .seed_data import seed_rows

# --------------------------------------------------------------------------------------
# helpers
# --------------------------------------------------------------------------------------

def _earliest(text: str, rules: List[Tuple[str, str]], flags=0):
    """Apply the rule whose match starts earliest (longest match wins ties). Returns new text or None."""
    best = None
    for pat, rep in rules:
        m = re.search(pat, text, flags)
        if m and (best is None or (m.start(), -len(m.group(0))) < (best[0].start(), -len(best[0].group(0)))):
            best = (m, pat, rep)
    if best is None:
        return None
    m, pat, rep = best
    return text[: m.start()] + m.expand(rep) + text[m.end():]


def _tidy(text: str, lang: str) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\s+([,.।])", r"\1", text)
    text = re.sub(r",\s*([.।])", r"\1", text)
    text = re.sub(r"([.।])\s*([.।])", r"\1", text)
    if lang == "en":
        # capitalise sentence starts
        text = re.sub(r"(^|[.!?]\s+)([a-z])", lambda m: m.group(1) + m.group(2).upper(), text)
        if not text.endswith((".", "!", "?")):
            text += "."
    else:
        if not text.endswith("।"):
            text += "।"
    return text


def _match_case(src: str, rep: str) -> str:
    return rep[:1].upper() + rep[1:] if src[:1].isupper() and rep else rep


def _split_sentences(text: str, lang: str) -> List[str]:
    if lang == "en":
        return [s for s in re.split(r"(?<!Rs\.)(?<=[.!?])\s+", text.strip()) if s]
    return [s.strip() + "।" for s in text.strip().split("।") if s.strip()]


# --------------------------------------------------------------------------------------
# perturbations: f(text, lang, domain, rng) -> Optional[str]
# --------------------------------------------------------------------------------------

def obligation_flip(text, lang, domain=None, rng=None):
    """must/shall <-> may/can  |  ना होगा <-> सकता है"""
    if lang == "en":
        rules = [
            (r"\bmust not\b", "may"), (r"\bshall not\b", "may"),
            (r"\bmust\b", "may"), (r"\bshall\b", "may"), (r"\bhas to\b", "may"),
            (r"\bmay\b", "must"), (r"\bcan\b", "must"),
        ]
    else:
        rules = [
            (r"(\S+?)ना होगा", r"\1 सकता है"), (r"(\S+?)नी होगी", r"\1 सकती है"),
            (r"(\S+?)ने होंगे", r"\1 सकता है"), (r"(\S+?)ना पड़ेगा", r"\1 सकता है"),
            (r"(\S+) सकता है", r"\1ना होगा"), (r"(\S+) सकती है", r"\1नी होगी"),
        ]
    out = _earliest(text, rules)
    return _tidy(out, lang) if out else None


def negation_flip(text, lang, domain=None, rng=None):
    """Remove the first negation, or insert one if there is none -> contradiction."""
    if lang == "en":
        if re.search(r"(?<!If )\bnot\b", text):
            out = re.sub(r"\s*(?<!If )\bnot\b", "", text, count=1)
        else:
            out = _earliest(text, [(r"\b(will|must|may|is|are|shall)\b", r"\1 not"), (r"\bcan\b", "cannot")])
    else:
        if "नहीं" in text:
            out = re.sub(r"नहीं\s*", "", text, count=1)
        else:
            out = _earliest(text, [
                (r"(\S+) (सकता|सकती) है", r"नहीं \1 \2"),
                (r" (होगा|होगी|होंगे|है|जाएगा|जाएगी)(?=[\s,।])", r" नहीं \1"),
            ])
    return _tidy(out, lang) if out and out != text else None


def drop_exception(text, lang, domain=None, rng=None):
    if lang == "en":
        rules = [
            (r",?\s*\bunless\b[^,.;]*", ""), (r",?\s*\bprovided( that)?\b[^,.;]*", ""),
            (r",?\s*\bexcept\b[^,.;]*", ""), (r"\s*\bwithout\b[^,.;]*", ""),
        ]
    else:
        rules = [
            (r",?\s*सिवाय इसके कि[^।]*", ""), (r",?\s*बशर्ते[^।]*", ""),
            (r",?\s*जब तक कि[^।]*", ""), (r"\S+ की (?:\S+ )?\S+ के बिना ", ""),
        ]
    out = _earliest(text, rules)
    return _tidy(out, lang) if out and out != text else None


def drop_condition(text, lang, domain=None, rng=None):
    if lang == "en":
        rules = [(r"\b(If|When) [^,]*,\s*", ""), (r"\s+\bif\b[^,.]*", "")]
    else:
        rules = [
            (r"(यदि|अगर) [^,]*, तो\s*", ""),
            (r"(?:(?<=^)|(?<=। ))(?:\S+ ){0,3}\S+ हों तो ", ""),
            (r"^[^,।]*? होने पर ", ""),
        ]
    out = _earliest(text, rules)
    return _tidy(out, lang) if out and out != text else None


_EN_NUM_WORDS = {"two": "four", "three": "six", "eleven": "six", "twelve": "six", "double": "half"}
_HI_NUM_WORDS = {"दो": "चार", "तीन": "छह", "ग्यारह": "छह", "बारह": "छह", "दोगुना": "आधा", "दोगुने": "आधे"}


def _indian_format(n: int) -> str:
    s = str(n)
    if len(s) <= 3:
        return s
    head, tail = s[:-3], s[-3:]
    parts = []
    while len(head) > 2:
        parts.insert(0, head[-2:])
        head = head[:-2]
    if head:
        parts.insert(0, head)
    return ",".join(parts + [tail])


def change_number(text, lang, domain=None, rng=None):
    def num_rep(m):
        raw = m.group(0)
        n = int(raw.replace(",", ""))
        new = n + 1 if 1900 <= n <= 2100 else n * 2
        return _indian_format(new) if "," in raw else str(new)

    cands = []
    m = re.search(r"\d[\d,]*\d|\d", text)
    if m:
        cands.append((m.start(), m, None))
    words = _EN_NUM_WORDS if lang == "en" else _HI_NUM_WORDS
    for w, rep in words.items():
        pat = rf"\b{w}\b" if lang == "en" else rf"(?<!\S){w}(?!\S)"
        mw = re.search(pat, text, re.I if lang == "en" else 0)
        if mw:
            cands.append((mw.start(), mw, rep))
    if not cands:
        return None
    _, m, rep = min(cands, key=lambda c: c[0])
    new = num_rep(m) if rep is None else _match_case(m.group(0), rep)
    return _tidy(text[: m.start()] + new + text[m.end():], lang)


def change_scope(text, lang, domain=None, rng=None):
    if lang == "en":
        rules = [
            (r"\b[Aa]ny\b", "some"), (r"\b[Aa]ll\b", "some"), (r"\b[Ff]ull\b", "partial"),
            (r"\b[Ee]ach\b", "the first"), (r"\b[Ee]very\b", "the first"),
            (r"\bonly\s+", ""), (r"\bvehicle\b", "car"), (r"\bEither side\b", "Only the landlord"),
        ]
        m_first = None
        for pat, _ in rules:
            m = re.search(pat, text)
            if m and (m_first is None or m.start() < m_first.start()):
                m_first = m
        out = _earliest(text, rules)
        if out and m_first is not None and m_first.group(0)[:1].isupper():
            out = out[: m_first.start()] + out[m_first.start():][:1].upper() + out[m_first.start() + 1:]
    else:
        rules = [
            ("किसी भी", "कुछ"), ("कोई भी", "कुछ"), ("सभी", "कुछ"), ("पूरा", "आंशिक"),
            ("(?<!\\S)हर ", "केवल पहले "), ("केवल ", ""), ("वाहन", "कार"),
        ]
        out = _earliest(text, rules)
    return _tidy(out, lang) if out and out != text else None


def omission(text, lang, domain=None, rng=None):
    sents = _split_sentences(text, lang)
    if len(sents) >= 2:
        return _tidy(" ".join(sents[:-1]), lang)
    # Otherwise drop the last comma-separated clause (", " so Indian-format numbers survive),
    # but never a clause that is an exception/condition (those have their own perturbations)
    # and never the main clause.
    if ", " in text:
        head, tail = text.rsplit(", ", 1)
        tail_s = tail.strip()
        is_exc = re.match(r"(unless|provided|except|सिवाय|बशर्ते|जब तक|तो )", tail_s)
        if lang == "en":
            head_is_condition = re.match(r"(If|When)\b", head.strip())
            if len(tail_s.split()) >= 3 and not is_exc and not head_is_condition:
                return _tidy(head, lang)
        else:
            tail_has_verb = re.search(r"(होगा|होगी|होंगे|है|सकता|सकती|जाएगा|जाएगी|करेगा)\s*।?$", tail_s)
            if len(tail_s.split()) >= 2 and not is_exc and not tail_has_verb:
                return _tidy(head, lang)
            # drop the middle item of a list "A, B या C"
            out = re.sub(r", (?!तो )([^,]+?) या ", " या ", text, count=1)
            if out != text:
                return _tidy(out, lang)
    return None


_HALLUCINATIONS = {
    ("en", "insurance"): [
        "The insurer will also cover any legal costs connected with the claim.",
        "This cover also applies to damage caused by earthquakes.",
        "The insured can claim this benefit twice a year.",
    ],
    ("en", "tenancy"): [
        "The tenant may also sublet one room without asking the landlord.",
        "The landlord will also pay the tenant's electricity bills.",
        "This rule does not apply during the first three months.",
    ],
    ("hi", "insurance"): [
        "बीमाकर्ता दावे से जुड़े सभी कानूनी खर्च भी उठाएगा।",
        "यह कवर भूकंप से हुए नुकसान पर भी लागू होता है।",
        "बीमाधारक साल में दो बार यह लाभ ले सकता है।",
    ],
    ("hi", "tenancy"): [
        "किरायेदार मकान-मालिक से पूछे बिना एक कमरा किराये पर दे सकता है।",
        "मकान-मालिक किरायेदार का बिजली बिल भी भरेगा।",
        "यह नियम पहले तीन महीनों में लागू नहीं होता।",
    ],
}


def hallucination(text, lang, domain="insurance", rng=None):
    rng = rng or random.Random(0)
    extra = rng.choice(_HALLUCINATIONS[(lang, domain)])
    return _tidy(text + " " + extra, lang)


def entity_swap(text, lang, domain=None, rng=None):
    if lang == "en":
        swap = {"insured": "insurer", "insurer": "insured", "tenant": "landlord",
                "landlord": "tenant", "policyholder": "insurer"}
        out = re.sub(r"\b(the|The) (insured|insurer|tenant|landlord|policyholder)\b",
                     lambda m: f"{m.group(1)} {swap[m.group(2)]}", text)
    else:
        swap = {"बीमाधारक": "बीमाकर्ता", "बीमाकर्ता": "बीमाधारक", "किरायेदार": "मकान-मालिक",
                "मकान-मालिक": "किरायेदार", "पॉलिसीधारक": "बीमाकर्ता"}
        out = re.sub(r"(बीमाधारक|बीमाकर्ता|पॉलिसीधारक|मकान-मालिक|किरायेदार(?!ी))",
                     lambda m: swap[m.group(1)], text)
    return _tidy(out, lang) if out != text else None


def benign_paraphrase(text, lang, domain=None, rng=None):
    """Meaning-preserving rewording (should keep a high LMP)."""
    if lang == "en":
        rules = [(r"\bmust not\b", "is not allowed to"), (r"\bmust\b", "has to"),
                 (r"\bwithin\b", "no later than"), (r"\bwritten notice\b", "notice in writing"),
                 (r"\bat least\b", "not less than"), (r"\bthe insurer\b", "the insurance company")]
    else:
        rules = [(r"ना होगा", "ना पड़ेगा"), (r"नी होगी", "नी पड़ेगी"), (r"ने होंगे", "ने पड़ेंगे"),
                 (r"अंदर", "भीतर"), (r"खत्म", "समाप्त"), (r"नुकसान", "हानि")]
    out = text
    for pat, rep in rules:
        out = re.sub(pat, rep, out)
    return _tidy(out, lang) if out != text else None


# name -> (function, error type, silver severity on the 1-10 scale)
PERTURBATIONS: Dict[str, Tuple[Callable, str, float]] = {
    "negation_flip": (negation_flip, "contradiction", 2.0),
    "entity_swap": (entity_swap, "entity", 2.5),
    "change_number": (change_number, "number_date", 3.5),
    "obligation_flip": (obligation_flip, "obligation_permission", 4.0),
    "drop_exception": (drop_exception, "exception", 4.0),
    "drop_condition": (drop_condition, "condition", 4.5),
    "hallucination": (hallucination, "hallucination", 4.5),
    "change_scope": (change_scope, "scope", 5.0),
    "omission": (omission, "omission", 5.0),
}


# --------------------------------------------------------------------------------------
# dataset builders
# --------------------------------------------------------------------------------------

def _clip(x):
    return float(np.clip(x, 1.0, 10.0))


def build_silver_dataset(seed: int = 13, doubles_per_source: int = 1) -> pd.DataFrame:
    """Faithful + benign + single + double perturbations for every seed clause."""
    rng = random.Random(seed)
    nrng = np.random.default_rng(seed)
    out = []

    def add(row, simp, score, errors, pert):
        out.append({**{k: row[k] for k in ("source_id", "concept_id", "lang", "domain", "context", "original")},
                    "simplified": simp, "lmp": round(_clip(score), 2),
                    "errors": ";".join(errors), "perturbation": pert,
                    "is_sanity": False, "label_source": "silver"})

    for row in seed_rows():
        lang, dom, faithful = row["lang"], row["domain"], row["faithful"]
        add(row, faithful, nrng.normal(9.3, 0.4), [], "faithful")
        b = benign_paraphrase(faithful, lang, dom, rng)
        if b:
            add(row, b, nrng.normal(8.8, 0.5), [], "benign")
        applicable = []
        for name, (fn, etype, sev) in PERTURBATIONS.items():
            p = fn(faithful, lang, dom, rng)
            if p and p != faithful:
                applicable.append(name)
                add(row, p, sev + nrng.normal(0, 0.5), [etype], name)
        for _ in range(doubles_per_source):
            if len(applicable) < 2:
                break
            a, c = rng.sample(applicable, 2)
            fa, ea, sa = PERTURBATIONS[a]
            fc, ec, sc = PERTURBATIONS[c]
            p1 = fa(faithful, lang, dom, rng)
            p2 = fc(p1, lang, dom, rng) if p1 else None
            if p2 and p2 != p1:
                add(row, p2, min(sa, sc) - 1.0 + nrng.normal(0, 0.4), sorted({ea, ec}), f"{a}+{c}")
    df = pd.DataFrame(out)
    df.insert(0, "pair_id", [f"P{i:04d}" for i in range(len(df))])
    return df


def build_sanity_pairs(df: pd.DataFrame, seed: int = 7) -> pd.DataFrame:
    """Identical pairs (expected LMP 10) and unrelated pairs (expected LMP 1).

    Unrelated pairs follow the JUDGEBERT paper: both sides are legal text with a shared
    lexicon (same language, same domain) but describe different clauses.
    """
    rng = random.Random(seed)
    src = df.drop_duplicates("source_id")[["source_id", "concept_id", "lang", "domain", "context", "original"]]
    pool = df[df.perturbation.isin(["faithful"])] if "perturbation" in df else df
    rows = []
    for _, r in src.iterrows():
        rows.append({**r.to_dict(), "simplified": r["original"], "lmp": 10.0, "errors": "",
                     "perturbation": "identical", "is_sanity": True, "label_source": "rule"})
        cands = pool[(pool.lang == r.lang) & (pool.domain == r.domain) & (pool.concept_id != r.concept_id)]
        if len(cands) == 0:
            cands = pool[(pool.lang == r.lang) & (pool.concept_id != r.concept_id)]
        if len(cands):
            other = cands.iloc[rng.randrange(len(cands))]
            rows.append({**r.to_dict(), "simplified": other["simplified"], "lmp": 1.0,
                         "errors": "omission;hallucination", "perturbation": "unrelated",
                         "is_sanity": True, "label_source": "rule"})
    out = pd.DataFrame(rows)
    out.insert(0, "pair_id", [f"S{i:04d}" for i in range(len(out))])
    return out


def perturbation_suite(base: pd.DataFrame) -> pd.DataFrame:
    """For every preserved (faithful) pair in `base`, create one variant per perturbation type.

    Works on human-annotated data too: pass the pairs whose human LMP >= 7.
    """
    rng = random.Random(0)
    rows = []
    for _, r in base.iterrows():
        rows.append({**r.to_dict(), "variant": "faithful"})
        for name, (fn, etype, _) in PERTURBATIONS.items():
            p = fn(r["simplified"], r["lang"], r["domain"], rng)
            if p and p != r["simplified"]:
                rows.append({**r.to_dict(), "simplified": p, "variant": name, "errors": etype})
    return pd.DataFrame(rows)


In [ ]:
%%writefile lmp/legal_diff.py
"""Rule-based legal change detector (English + Hindi).

Compares the *legal signature* of the original and the simplified text:
modality (obligation / permission / prohibition), negation, exceptions, conditions,
quantifiers (scope), numbers/dates and legal parties (entities).

It is used for
  * human-readable evidence in the explanation of each prediction, and
  * a transparent rule baseline for the legal-error-type classification (Exp. 5).
"""
import re
from collections import Counter
from typing import Dict, List

LEX = {
    "en": {
        "obligation": [r"\bshall\b(?! not)", r"\bmust\b(?! not)", r"\brequired to\b", r"\bhas to\b", r"\bhave to\b",
                       r"\bliable to\b", r"\bresponsible for\b", r"\bbound to\b", r"\bwill\b(?! not)"],
        "permission": [r"\bmay\b(?! not)", r"\bcan\b", r"\bentitled to\b", r"\ballowed to\b(?<!not allowed to)"],
        "negation": [r"(?<!if )(?<!does )\bnot\b(?! less than)", r"\bno\b(?! later| less)", r"\bnever\b", r"\bcannot\b", r"\bnothing\b"],
        "exception": [r"\bunless\b", r"\bexcept\b", r"\bprovided\b", r"\bsave\b", r"\bsubject to\b",
                      r"\bwithout\b", r"\bother than\b", r"\bfailing which\b", r"\bfailure to\b", r"\botherwise\b",
                      r"\bif (?:he |she |they |it |you )?(?:does |do )?not\b"],
        "condition": [r"\bif\b", r"\bwhere\b", r"\bin the event\b", r"\bin case\b", r"\bwhen\b"],
        "scope": [r"\bany\b", r"\ball\b", r"\beach\b", r"\bevery\b", r"\bonly\b", r"\bsolely\b", r"\bfull\b",
                  r"\bin full\b", r"\beither\b", r"\bsome\b", r"\bpartial\b", r"\bpart of\b", r"\bfirst\b"],
        "entity": {"insured": r"\binsured\b(?! property| vehicle)|\bpolicyholder\b",
                   "insurer": r"\binsurer\b|\binsurance company\b",
                   "tenant": r"\btenant\b", "landlord": r"\blandlord\b", "police": r"\bpolice\b"},
    },
    "hi": {
        "obligation": [r"ना होगा", r"नी होगी", r"ने होंगे", r"ना पड़ेगा", r"नी पड़ेगी", r"ने पड़ेंगे",
                       r"करेगा", r"करेगी", r"देगा", r"उत्तरदायी", r"दायी", r"बाध्य", r"अनिवार्य", r"जाएगा", r"जाएगी"],
        "permission": [r"सकता", r"सकती", r"सकते", r"हकदार", r"अनुमति"],
        "negation": [r"नहीं", r"(?<!\S)न ही(?!\S)"],
        "exception": [r"सिवाय", r"बशर्ते", r"जब तक कि", r"के बिना", r"अन्यथा", r"न करने पर"],
        "condition": [r"यदि", r"अगर", r"की स्थिति में", r"होने पर", r"जहाँ", r"(?<!\S)तो(?!\S)", r"हों तो"],
        "scope": [r"किसी भी", r"कोई भी", r"सभी", r"प्रत्येक", r"(?<!\S)हर(?!\S)", r"केवल", r"पूरा", r"पूर्ण",
                  r"(?<!\S)कुछ(?!\S)", r"आंशिक"],
        "entity": {"insured": r"बीमाधारक|पॉलिसीधारक", "insurer": r"बीमाकर्ता", "tenant": r"किरायेदार(?!ी)",
                   "landlord": r"मकान-मालिक", "police": r"पुलिस"},
    },
}

NUM_WORDS = {
    "en": {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "eleven": 11, "twelve": 12,
           "fifteen": 15, "thirty": 30, "double": "x2", "twice": "x2", "half": "x0.5"},
    "hi": {"एक": 1, "दो": 2, "तीन": 3, "चार": 4, "पाँच": 5, "छह": 6, "ग्यारह": 11, "बारह": 12,
           "दोगुना": "x2", "दोगुने": "x2", "आधा": "x0.5", "आधे": "x0.5"},
}


def _count(text: str, pats: List[str]) -> int:
    return sum(len(re.findall(p, text, flags=re.I)) for p in pats)


def _numbers(text: str, lang: str) -> Counter:
    nums = [re.sub(r",", "", m) for m in re.findall(r"\d[\d,]*\d|\d", text)]
    words = NUM_WORDS[lang]
    for w, v in words.items():
        pat = rf"\b{w}\b" if lang == "en" else rf"(?<!\S){w}(?!\S)"
        nums += [str(v)] * len(re.findall(pat, text, flags=re.I if lang == "en" else 0))
    return Counter(nums)


def _scope_words(text: str, lang: str) -> Counter:
    out = Counter()
    for p in LEX[lang]["scope"]:
        for m in re.findall(p, text, flags=re.I):
            out[m.lower().strip()] += 1
    # normalise near-synonyms so original vs plain-language wording is not flagged
    norm = {"solely": "only", "in full": "full", "पूर्ण": "पूरा", "प्रत्येक": "हर", "किसी भी": "any", "कोई भी": "any",
            "each": "every"}
    return Counter({norm.get(k, k): v for k, v in out.items()})


def _entities(text: str, lang: str) -> List[str]:
    hits = []
    for name, pat in LEX[lang]["entity"].items():
        for m in re.finditer(pat, text, flags=re.I):
            hits.append((m.start(), name))
    return [n for _, n in sorted(hits)]


def signature(text: str, lang: str) -> Dict:
    L = LEX[lang]
    return {
        "obligation": _count(text, L["obligation"]),
        "permission": _count(text, L["permission"]),
        "negation": _count(text, L["negation"]),
        "exception": _count(text, L["exception"]),
        "condition": _count(text, L["condition"]),
        "numbers": _numbers(text, lang),
        "scope": _scope_words(text, lang),
        "entities": _entities(text, lang),
        "n_tokens": len(text.split()),
        "n_sents": max(1, len(re.findall(r"(?<!Rs)[.!?।](?:\s|$)", text))),
    }


def legal_diff(original: str, simplified: str, lang: str) -> Dict:
    """Return detected legal changes as {'findings': [...], 'error_types': set(...)}."""
    a, b = signature(original, lang), signature(simplified, lang)
    findings, types = [], set()

    # numbers / dates
    missing = a["numbers"] - b["numbers"]
    added = b["numbers"] - a["numbers"]
    if missing or added:
        types.add("number_date")
        findings.append(f"Numbers/dates differ: original has {sorted(missing.elements()) or '-'}, "
                        f"simplified has {sorted(added.elements()) or '-'}.")

    # negation polarity
    if (a["negation"] % 2) != (b["negation"] % 2) or (a["negation"] > 0) != (b["negation"] > 0):
        types.add("contradiction")
        findings.append(f"Negation polarity changed ({a['negation']} negation(s) in original vs {b['negation']} in simplified).")

    # modality
    if a["obligation"] > 0 and b["obligation"] == 0 and b["permission"] > 0:
        types.add("obligation_permission")
        findings.append("An obligation (shall/must) became a permission (may/can).")
    elif a["obligation"] == 0 and a["permission"] > 0 and b["obligation"] > 0 and b["permission"] == 0:
        types.add("obligation_permission")
        findings.append("A permission (may/can) became an obligation (shall/must).")
    elif b["permission"] > a["permission"] and b["obligation"] < a["obligation"]:
        types.add("obligation_permission")
        findings.append("Modality shifted from obligation towards permission.")

    if a["exception"] > 0 and b["exception"] == 0:
        types.add("exception")
        findings.append("An exception/proviso (unless, provided that, without ...) was dropped.")
    if a["condition"] > 0 and b["condition"] == 0:
        types.add("condition")
        findings.append("A condition (if, where, in the event of ...) was dropped.")

    # scope: flag a universal quantifier replaced by a restrictive one, or an added/removed "only"
    sa, sb = a["scope"], b["scope"]
    universal = {"any", "all", "every", "full", "either", "हर", "सभी", "पूरा"}
    restrictive = {"some", "partial", "part of", "first", "कुछ", "आंशिक"}
    lost, gained = set(sa) - set(sb), set(sb) - set(sa)
    only_changed = ({"only", "केवल"} & (lost | gained))
    if (lost & universal and gained & restrictive) or only_changed:
        types.add("scope")
        findings.append(f"Scope changed: lost {sorted(lost) or '-'}, gained {sorted(gained) or '-'}.")

    # entities / party roles: same parties, but a different party is mentioned first (role swap)
    ea, eb = a["entities"], b["entities"]
    if lang == "en" and ea and eb and set(ea) == set(eb) and len(set(ea)) > 1 and ea[0] != eb[0]:
        types.add("entity")
        findings.append(f"Party roles may be swapped: {ea} -> {eb}.")

    # length-based omission / hallucination hints
    ratio = b["n_tokens"] / max(a["n_tokens"], 1)
    if ratio < 0.45:
        types.add("omission")
        findings.append(f"Simplified text is much shorter ({ratio:.0%} of the original length).")
    elif ratio > 1.6 or b["n_sents"] >= a["n_sents"] + 2:
        types.add("hallucination")
        findings.append(f"Simplified text is much longer ({ratio:.0%} of the original length) - possible added content.")

    return {"findings": findings, "error_types": types, "orig_sig": a, "simp_sig": b}


In [ ]:
%%writefile lmp/data.py
"""Loading human-annotated data, aggregating annotators, splitting and augmentation."""
import re
from typing import Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

from .config import Config
from .perturb import build_sanity_pairs, build_silver_dataset

REQUIRED = ["lang", "domain", "original", "simplified"]


def krippendorff_alpha(df: pd.DataFrame, cols) -> float:
    """Interval Krippendorff's alpha over annotator columns (NaN = not annotated)."""
    try:
        import krippendorff
        data = df[cols].T.to_numpy(dtype=float)
        return float(krippendorff.alpha(reliability_data=data, level_of_measurement="interval"))
    except Exception as e:  # pragma: no cover
        print("krippendorff not available:", e)
        return float("nan")


def load_annotated_csv(path: str) -> pd.DataFrame:
    """Load the human-annotated dataset (see annotation_template.csv).

    Columns: pair_id, source_id, lang, domain, context, original, simplified,
             annotator_1 ... annotator_k (1-10), errors (optional, ';'-separated), notes
    The gold LMP is the mean of the annotator columns (as in the JUDGEBERT paper).
    """
    df = pd.read_csv(path)
    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        raise ValueError(f"CSV is missing columns: {missing}")
    ann = [c for c in df.columns if re.match(r"annotator_\d+", c)]
    if "lmp" not in df.columns:
        if not ann:
            raise ValueError("Need either an 'lmp' column or annotator_1..k columns.")
        df["lmp"] = df[ann].mean(axis=1)
    if ann:
        print(f"Annotators: {len(ann)} | Krippendorff's alpha (interval): {krippendorff_alpha(df, ann):.3f}")
        df["lmp_std"] = df[ann].std(axis=1)
    df = df.dropna(subset=["lmp"]).copy()
    if "pair_id" not in df.columns:
        df["pair_id"] = [f"H{i:04d}" for i in range(len(df))]
    if "source_id" not in df.columns:
        df["source_id"] = df.groupby(["lang", "original"]).ngroup().map(lambda i: f"SRC{i:04d}")
    if "concept_id" not in df.columns:
        df["concept_id"] = df["source_id"]
    if "context" not in df.columns:
        df["context"] = ""
    if "errors" not in df.columns:
        df["errors"] = np.nan          # NaN = error labels unknown -> masked in the loss
    df["context"] = df["context"].fillna("")
    df["perturbation"] = df.get("perturbation", "human")
    df["is_sanity"] = False
    df["label_source"] = "human"
    return df


def prepare_data(cfg: Config, human_csv: Optional[str] = None, silver_seed: int = 13) -> pd.DataFrame:
    """Main dataset: human CSV if given, otherwise the silver seed dataset."""
    if human_csv:
        df = load_annotated_csv(human_csv)
        print(f"Loaded {len(df)} human-annotated pairs.")
    else:
        df = build_silver_dataset(seed=silver_seed)
        print(f"No human CSV given -> using {len(df)} SILVER pairs (pipeline demo only, not paper results).")
    return df.reset_index(drop=True)


def split_data(df: pd.DataFrame, cfg: Config, seed: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Grouped split: all variants of one clause (and its translation) stay in the same split.

    The JUDGEBERT paper split instances randomly; grouping avoids leakage where a perturbed
    copy of a test clause appears in training.
    """
    groups = df["concept_id"] if "concept_id" in df else df["source_id"]
    gss = GroupShuffleSplit(n_splits=1, test_size=cfg.test_size, random_state=seed)
    tr_idx, te_idx = next(gss.split(df, groups=groups))
    train_val, test = df.iloc[tr_idx], df.iloc[te_idx]
    g2 = train_val["concept_id"] if "concept_id" in train_val else train_val["source_id"]
    gss2 = GroupShuffleSplit(n_splits=1, test_size=cfg.val_size / (1 - cfg.test_size), random_state=seed)
    tr2, va2 = next(gss2.split(train_val, groups=g2))
    return (train_val.iloc[tr2].reset_index(drop=True),
            train_val.iloc[va2].reset_index(drop=True),
            test.reset_index(drop=True))


def augment(train: pd.DataFrame, cfg: Config, extra_silver: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    """Add identical (10) / unrelated (1) sanity pairs built ONLY from training clauses (JUDGEBERT-DA)."""
    parts = [train]
    if cfg.data_augmentation:
        parts.append(build_sanity_pairs(train, seed=11))
    if extra_silver is not None and len(extra_silver):
        parts.append(extra_silver)
    return pd.concat(parts, ignore_index=True)


def errors_to_multihot(value, error_types) -> Tuple[np.ndarray, float]:
    """Returns (multi-hot vector, mask). mask=0 when error labels are unknown (NaN)."""
    vec = np.zeros(len(error_types), dtype=np.float32)
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return vec, 0.0
    for e in str(value).split(";"):
        e = e.strip()
        if e in error_types:
            vec[error_types.index(e)] = 1.0
    return vec, 1.0


In [ ]:
%%writefile lmp/model.py
"""Explainable multi-task cross-encoder for Legal Meaning Preservation (LMP).

Like JUDGEBERT, the original and simplified texts are concatenated with a separator and
fed to one transformer. Unlike JUDGEBERT (single regression head, French-only CamemBERT),
this model is multilingual and has three heads on a shared encoder:
    score head    -> LMP in [1, 10]
    decision head -> Not / Partially / Preserved
    error head    -> multi-label legal error types (omission, obligation_permission, ...)
"""
import torch
import torch.nn as nn
from transformers import AutoModel

from .config import DECISIONS, ERROR_TYPES


class LMPModel(nn.Module):
    def __init__(self, model_name: str, dropout: float = 0.1, n_errors: int = len(ERROR_TYPES)):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.score_head = nn.Sequential(nn.Linear(h, h // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(h // 2, 1))
        self.decision_head = nn.Linear(h, len(DECISIONS))
        self.error_head = nn.Linear(h, n_errors)

    def pool(self, hidden, mask):
        mask = mask.unsqueeze(-1).to(hidden.dtype)
        return (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-6)

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, inputs_embeds=None):
        kwargs = {"attention_mask": attention_mask}
        if inputs_embeds is not None:
            kwargs["inputs_embeds"] = inputs_embeds
        else:
            kwargs["input_ids"] = input_ids
        if token_type_ids is not None and getattr(self.encoder.config, "type_vocab_size", 1) > 1:
            kwargs["token_type_ids"] = token_type_ids
        hidden = self.encoder(**kwargs).last_hidden_state
        pooled = self.dropout(self.pool(hidden, attention_mask))
        raw = self.score_head(pooled).squeeze(-1)
        score = 1.0 + 9.0 * torch.sigmoid(raw)       # bounded to the 1-10 Likert range
        return {"score": score, "decision_logits": self.decision_head(pooled),
                "error_logits": self.error_head(pooled)}


In [ ]:
%%writefile lmp/train.py
"""Training / inference for the LMP cross-encoder (plain PyTorch loop, works on Colab GPU or CPU)."""
import copy
import math
import random
import time
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

from .config import DECISIONS, ERROR_TYPES, Config, score_to_decision_id
from .data import errors_to_multihot
from .model import LMPModel


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device(cfg: Config) -> torch.device:
    if cfg.device:
        return torch.device(cfg.device)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def first_input(row, cfg: Config) -> str:
    """Segment A = [context] + original. Segment B = simplified."""
    ctx = str(row.get("context", "") or "")
    if cfg.use_context and ctx.strip():
        return f"{ctx} || {row['original']}"
    return str(row["original"])


class PairDataset(Dataset):
    def __init__(self, df: pd.DataFrame, cfg: Config):
        self.rows = df.to_dict("records")
        self.cfg = cfg

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        err, mask = errors_to_multihot(r.get("errors", np.nan), ERROR_TYPES)
        lmp = float(r.get("lmp", np.nan)) if r.get("lmp") is not None else np.nan
        return {
            "a": first_input(r, self.cfg), "b": str(r["simplified"]),
            "lmp": lmp,
            "decision": score_to_decision_id(lmp, self.cfg) if not np.isnan(lmp) else -100,
            "errors": err, "err_mask": mask,
        }


def make_collate(tokenizer, cfg: Config):
    def collate(batch):
        enc = tokenizer([b["a"] for b in batch], [b["b"] for b in batch], truncation=True,
                        max_length=cfg.max_len, padding=True, return_tensors="pt")
        enc["lmp"] = torch.tensor([b["lmp"] for b in batch], dtype=torch.float)
        enc["decision"] = torch.tensor([b["decision"] for b in batch], dtype=torch.long)
        enc["errors"] = torch.tensor(np.stack([b["errors"] for b in batch]))
        enc["err_mask"] = torch.tensor([b["err_mask"] for b in batch], dtype=torch.float)
        return enc
    return collate


def multitask_loss(out, batch, cfg: Config, pos_weight=None) -> Tuple[torch.Tensor, Dict[str, float]]:
    pred, gold = out["score"].float(), batch["lmp"]
    diff = pred - gold
    # asymmetric MSE: over-estimation (approving a bad simplification) costs more
    w = torch.where(diff > 0, torch.full_like(diff, cfg.over_penalty), torch.ones_like(diff))
    l_score = (w * diff.pow(2)).mean() / cfg.over_penalty
    l_dec = F.cross_entropy(out["decision_logits"].float(), batch["decision"], ignore_index=-100)
    # pos_weight balances rare error types (each type occurs in only a small share of pairs)
    bce = F.binary_cross_entropy_with_logits(out["error_logits"].float(), batch["errors"], reduction="none",
                                             pos_weight=pos_weight).mean(1)
    l_err = (bce * batch["err_mask"]).sum() / batch["err_mask"].sum().clamp(min=1.0)
    total = l_score + cfg.w_decision * l_dec + cfg.w_errors * l_err
    return total, {"score": l_score.item(), "decision": l_dec.item(), "errors": l_err.item()}


@torch.no_grad()
def predict(model: LMPModel, tokenizer, df: pd.DataFrame, cfg: Config, batch_size: int = 64) -> pd.DataFrame:
    device = next(model.parameters()).device
    model.eval()
    tmp = df.copy()
    if "lmp" not in tmp:
        tmp["lmp"] = np.nan
    dl = DataLoader(PairDataset(tmp, cfg), batch_size=batch_size, shuffle=False,
                    collate_fn=make_collate(tokenizer, cfg))
    scores, dec, errs = [], [], []
    for batch in dl:
        inputs = {k: batch[k].to(device) for k in ("input_ids", "attention_mask", "token_type_ids") if k in batch}
        out = model(**inputs)
        scores.append(out["score"].float().cpu().numpy())
        dec.append(out["decision_logits"].float().softmax(-1).cpu().numpy())
        errs.append(torch.sigmoid(out["error_logits"].float()).cpu().numpy())
    scores, dec, errs = np.concatenate(scores), np.concatenate(dec), np.concatenate(errs)
    res = pd.DataFrame({"pred_score": scores, "pred_decision": [DECISIONS[i] for i in dec.argmax(1)]}, index=df.index)
    for j, e in enumerate(ERROR_TYPES):
        res[f"p_{e}"] = errs[:, j]
    return res


def _quick_eval(model, tokenizer, df, cfg) -> Dict[str, float]:
    p = predict(model, tokenizer, df, cfg)["pred_score"].to_numpy()
    y = df["lmp"].to_numpy(dtype=float)
    rmse = float(np.sqrt(np.mean((p - y) ** 2)))
    r = float(np.corrcoef(p, y)[0, 1]) if np.std(p) > 1e-8 and np.std(y) > 1e-8 else 0.0
    return {"rmse": rmse, "pearson": r}


def train_model(train_df: pd.DataFrame, val_df: pd.DataFrame, cfg: Config, seed: int = 42,
                verbose: bool = True):
    """Fine-tune the multi-task cross-encoder. Returns (model, tokenizer, history)."""
    set_seed(seed)
    device = get_device(cfg)
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
    model = LMPModel(cfg.model_name, cfg.dropout).to(device)

    dl = DataLoader(PairDataset(train_df, cfg), batch_size=cfg.batch_size, shuffle=True,
                    collate_fn=make_collate(tokenizer, cfg))
    enc_params = [p for n, p in model.named_parameters() if n.startswith("encoder.")]
    head_params = [p for n, p in model.named_parameters() if not n.startswith("encoder.")]
    opt = torch.optim.AdamW([{"params": enc_params, "lr": cfg.lr},
                             {"params": head_params, "lr": cfg.head_lr}], weight_decay=cfg.weight_decay)
    total = max(1, len(dl) * cfg.epochs)
    sched = get_linear_schedule_with_warmup(opt, int(cfg.warmup_ratio * total), total)
    use_amp = cfg.fp16 and device.type == "cuda"
    # class-balancing weights for the error-type head, estimated on rows with known error labels
    ev = [errors_to_multihot(v, ERROR_TYPES) for v in train_df.get("errors", pd.Series([np.nan] * len(train_df)))]
    known = np.array([v for v, m in ev if m > 0])
    if len(known):
        pos = known.sum(0)
        pos_weight = torch.tensor(np.clip((len(known) - pos) / np.maximum(pos, 1), 1.0, 5.0),
                                  dtype=torch.float, device=device)
    else:
        pos_weight = None
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    best, best_state, bad, history = math.inf, None, 0, []
    for epoch in range(1, cfg.epochs + 1):
        model.train()
        t0, run = time.time(), []
        for batch in dl:
            batch = {k: v.to(device) for k, v in batch.items()}
            inputs = {k: batch[k] for k in ("input_ids", "attention_mask", "token_type_ids") if k in batch}
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                out = model(**inputs)
            loss, parts = multitask_loss(out, batch, cfg, pos_weight)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            run.append(loss.item())
        val = _quick_eval(model, tokenizer, val_df, cfg) if len(val_df) else {"rmse": float(np.mean(run)), "pearson": 0}
        history.append({"epoch": epoch, "train_loss": float(np.mean(run)), **{f"val_{k}": v for k, v in val.items()}})
        if verbose:
            print(f"  epoch {epoch:2d} | train loss {np.mean(run):.3f} | val RMSE {val['rmse']:.3f} "
                  f"| val r {val['pearson']:.3f} | {time.time() - t0:.0f}s")
        if epoch < cfg.min_epochs:
            continue                      # let the randomly initialised heads settle first
        if val["rmse"] < best - 1e-4:
            best, bad = val["rmse"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= cfg.patience:
                if verbose:
                    print(f"  early stopping at epoch {epoch}")
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    return model, tokenizer, pd.DataFrame(history)


def save_model(model: LMPModel, tokenizer, path: str):
    import json
    import os
    os.makedirs(path, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(path, "lmp_model.pt"))
    tokenizer.save_pretrained(path)
    model.encoder.config.save_pretrained(path)
    with open(os.path.join(path, "lmp_meta.json"), "w") as f:
        json.dump({"error_types": ERROR_TYPES, "decisions": DECISIONS}, f)


def load_model(path: str, cfg: Config):
    """Reload a saved model (encoder architecture is rebuilt from cfg.model_name)."""
    import os
    tokenizer = AutoTokenizer.from_pretrained(path)
    model = LMPModel(cfg.model_name, cfg.dropout)
    model.load_state_dict(torch.load(os.path.join(path, "lmp_model.pt"), map_location="cpu"))
    return model.to(get_device(cfg)).eval(), tokenizer


In [ ]:
%%writefile lmp/metrics.py
"""Evaluation metrics: correlation, error, over-estimation, decisions, error types, sanity, bootstrap CIs."""
from typing import Dict, List

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

from .config import DECISIONS, Config, score_to_decision


def _safe_corr(fn, y, p):
    if len(y) < 3 or np.std(y) < 1e-9 or np.std(p) < 1e-9:
        return float("nan")
    return float(fn(y, p)[0])


def regression_metrics(y, p, cfg: Config, tol: float = 0.5) -> Dict[str, float]:
    y, p = np.asarray(y, float), np.asarray(p, float)
    yd = [score_to_decision(v, cfg) for v in y]
    pd_ = [score_to_decision(v, cfg) for v in p]
    return {
        "pearson": _safe_corr(pearsonr, y, p),
        "spearman": _safe_corr(spearmanr, y, p),
        "rmse": float(np.sqrt(np.mean((p - y) ** 2))),
        "mae": float(np.mean(np.abs(p - y))),
        # % of predictions that over-estimate preservation by more than `tol` Likert levels
        # (the dangerous direction; JUDGEBERT paper Table 6)
        "over_%": float(100 * np.mean(p > y + tol)),
        "decision_acc": float(accuracy_score(yd, pd_)),
        "decision_macroF1": float(f1_score(yd, pd_, labels=DECISIONS, average="macro", zero_division=0)),
        "n": int(len(y)),
    }


def bootstrap_ci(y, p, cfg: Config, metric="pearson", n_boot: int = 1000, seed: int = 0):
    rng = np.random.default_rng(seed)
    y, p = np.asarray(y, float), np.asarray(p, float)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        v = regression_metrics(y[idx], p[idx], cfg)[metric]
        if not np.isnan(v):
            vals.append(v)
    if not vals:
        return float("nan"), float("nan")
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def error_type_metrics(gold: pd.Series, prob: pd.DataFrame, error_types: List[str], thr: float = 0.5) -> pd.DataFrame:
    """Per-type and micro/macro P/R/F1. `gold` = ';'-separated strings, `prob` = columns p_<type>."""
    mask = gold.notna()
    Y = np.array([[int(e in str(g).split(";")) for e in error_types] for g in gold[mask]])
    P = (prob.loc[mask, [f"p_{e}" for e in error_types]].to_numpy() >= thr).astype(int)
    rows = []
    for j, e in enumerate(error_types):
        pr, rc, f1, _ = precision_recall_fscore_support(Y[:, j], P[:, j], average="binary", zero_division=0)
        rows.append({"error_type": e, "precision": pr, "recall": rc, "f1": f1, "support": int(Y[:, j].sum())})
    for avg in ("micro", "macro"):
        pr, rc, f1, _ = precision_recall_fscore_support(Y, P, average=avg, zero_division=0)
        rows.append({"error_type": f"[{avg}]", "precision": pr, "recall": rc, "f1": f1, "support": int(Y.sum())})
    return pd.DataFrame(rows)


def sanity_metrics(scores: np.ndarray, kinds: np.ndarray) -> Dict[str, float]:
    """JUDGEBERT sanity checks on the 1-10 scale normalised to [0, 1]:
    identical pairs should score >= 0.99, unrelated pairs <= 0.01 (1% float margin, as in the paper)."""
    s = (np.asarray(scores, float) - 1.0) / 9.0
    kinds = np.asarray(kinds)
    ident, unrel = s[kinds == "identical"], s[kinds == "unrelated"]
    return {
        "identical_>=99%": float(100 * np.mean(ident >= 0.99)) if len(ident) else float("nan"),
        "unrelated_<=1%": float(100 * np.mean(unrel <= 0.01)) if len(unrel) else float("nan"),
        "identical_mean": float(1 + 9 * ident.mean()) if len(ident) else float("nan"),
        "unrelated_mean": float(1 + 9 * unrel.mean()) if len(unrel) else float("nan"),
    }


def summarize_seeds(per_seed: List[Dict[str, float]]) -> Dict[str, str]:
    """mean ± std over seeds, formatted for tables."""
    df = pd.DataFrame(per_seed)
    out = {}
    for c in df.columns:
        if c == "n":
            out[c] = int(df[c].iloc[0])
            continue
        m, s = df[c].mean(), df[c].std(ddof=0)
        out[c] = f"{m:.3f} ± {s:.3f}" if len(df) > 1 else f"{m:.3f}"
    return out


In [ ]:
%%writefile lmp/baselines.py
"""Generic semantic-similarity baselines, all mapped to the 1-10 LMP scale (decimal scaling, as in the paper).

Lexical : TF-IDF char n-gram cosine, chrF, BLEU
Neural  : BERTScore-F1 (multilingual XLM-R, own implementation), SBERT multilingual, LaBSE
"""
from typing import Dict

import numpy as np
import pandas as pd
import torch


def _to_scale(sim) -> np.ndarray:
    return 1.0 + 9.0 * np.clip(np.asarray(sim, dtype=float), 0.0, 1.0)


def tfidf_char(df: pd.DataFrame) -> np.ndarray:
    from sklearn.feature_extraction.text import TfidfVectorizer
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True)
    vec.fit(pd.concat([df["original"], df["simplified"]]).astype(str))
    A, B = vec.transform(df["original"].astype(str)), vec.transform(df["simplified"].astype(str))
    sims = np.asarray(A.multiply(B).sum(1)).ravel()   # rows are L2-normalised -> dot = cosine
    return _to_scale(sims)


def chrf(df: pd.DataFrame) -> np.ndarray:
    from sacrebleu.metrics import CHRF
    m = CHRF()
    return _to_scale([m.sentence_score(s, [o]).score / 100 for o, s in zip(df["original"], df["simplified"])])


def bleu(df: pd.DataFrame) -> np.ndarray:
    from sacrebleu.metrics import BLEU
    m = BLEU(tokenize="intl", effective_order=True)
    return _to_scale([m.sentence_score(s, [o]).score / 100 for o, s in zip(df["original"], df["simplified"])])


@torch.no_grad()
def bertscore_f1(df: pd.DataFrame, model_name="xlm-roberta-base", layer=9, batch_size=32) -> np.ndarray:
    """BERTScore F1 (Zhang et al., 2019) without IDF, greedy cosine matching of contextual embeddings."""
    from transformers import AutoModel, AutoTokenizer
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name, output_hidden_states=True).to(dev).eval()

    def embed(texts):
        enc = tok(list(texts), padding=True, truncation=True, max_length=256, return_tensors="pt",
                  return_special_tokens_mask=True)
        special = enc.pop("special_tokens_mask").bool()
        enc = {k: v.to(dev) for k, v in enc.items()}
        hs = model(**enc).hidden_states[layer]
        hs = torch.nn.functional.normalize(hs, dim=-1)
        mask = enc["attention_mask"].bool() & ~special.to(dev)
        return hs, mask

    out = []
    o, s = df["original"].astype(str).tolist(), df["simplified"].astype(str).tolist()
    for i in range(0, len(o), batch_size):
        ho, mo = embed(o[i:i + batch_size])
        hs, ms = embed(s[i:i + batch_size])
        sim = torch.bmm(hs, ho.transpose(1, 2))                      # [B, Ls, Lo]
        sim = sim.masked_fill(~mo.unsqueeze(1), -2).masked_fill(~ms.unsqueeze(2), -2)
        prec = (sim.max(2).values * ms).sum(1) / ms.sum(1).clamp(min=1)       # simplified -> original
        rec = (sim.max(1).values * mo).sum(1) / mo.sum(1).clamp(min=1)        # original -> simplified
        f1 = 2 * prec * rec / (prec + rec).clamp(min=1e-8)
        out.append(f1.cpu().numpy())
    del model
    return _to_scale(np.concatenate(out))


@torch.no_grad()
def sbert_cosine(df: pd.DataFrame, model_name: str) -> np.ndarray:
    from sentence_transformers import SentenceTransformer
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    m = SentenceTransformer(model_name, device=dev)
    a = m.encode(df["original"].astype(str).tolist(), normalize_embeddings=True, batch_size=64)
    b = m.encode(df["simplified"].astype(str).tolist(), normalize_embeddings=True, batch_size=64)
    del m
    return _to_scale((a * b).sum(1))


def run_baselines(df: pd.DataFrame, cfg, neural: bool = None) -> pd.DataFrame:
    """Return a DataFrame (same index as df) with one 1-10 score column per baseline."""
    neural = cfg.run_neural_baselines if neural is None else neural
    res: Dict[str, np.ndarray] = {}
    res["TF-IDF(char)"] = tfidf_char(df)
    res["chrF"] = chrf(df)
    res["BLEU"] = bleu(df)
    if neural:
        try:
            res["BERTScore(XLM-R)"] = bertscore_f1(df, cfg.bertscore_model, cfg.bertscore_layer)
        except Exception as e:
            print("BERTScore skipped:", e)
        for name in cfg.sbert_models:
            short = "LaBSE" if "LaBSE" in name else "SBERT-Multi"
            try:
                res[short] = sbert_cosine(df, name)
            except Exception as e:
                print(f"{short} skipped:", e)
    return pd.DataFrame(res, index=df.index)


In [ ]:
%%writefile lmp/explain.py
"""Explanations: Integrated-Gradients token attributions + rule-based legal evidence + error-type head."""
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch

from .config import ERROR_TYPES, Config
from .legal_diff import legal_diff
from .train import first_input, predict

ERROR_DESCRIPTIONS = {
    "omission": "an essential fact of the original is missing",
    "hallucination": "the simplification adds a fact that is not in the original",
    "contradiction": "the simplification states the opposite of the original (negation flipped)",
    "obligation_permission": "an obligation (shall/must) and a permission (may/can) were confused",
    "condition": "a condition (if / in the event of) was dropped or changed",
    "exception": "an exception or proviso (unless / provided that) was dropped or changed",
    "scope": "the scope changed (any/all/every/only/vehicle vs car ...)",
    "number_date": "an amount, time limit, percentage or date changed",
    "entity": "the legal parties/roles were changed or swapped",
}


def _merge_subwords(tokens: List[str], scores: np.ndarray) -> List[Tuple[str, float]]:
    words, cur, cur_s = [], "", 0.0
    for t, s in zip(tokens, scores):
        if t.startswith("▁") or t.startswith("Ġ") or (not t.startswith("##") and cur == ""):
            if cur:
                words.append((cur, cur_s))
            cur, cur_s = t.lstrip("▁Ġ"), float(s)
        elif t.startswith("##"):
            cur, cur_s = cur + t[2:], cur_s + float(s)
        else:
            if cur:
                words.append((cur, cur_s))
            cur, cur_s = t, float(s)
    if cur:
        words.append((cur, cur_s))
    return [(w, s) for w, s in words if w.strip()]


def integrated_gradients(model, tokenizer, original: str, simplified: str, cfg: Config,
                         context: str = "", steps: int = 32) -> Dict[str, List[Tuple[str, float]]]:
    """Attribution of the LMP score to each word (Sundararajan et al., 2017).

    Negative attribution = the word pushed the predicted preservation score DOWN.
    Baseline = all-<pad> embeddings (special tokens kept).
    """
    device = next(model.parameters()).device
    model.eval()
    a = first_input({"original": original, "context": context}, cfg)
    enc = tokenizer(a, simplified, truncation=True, max_length=cfg.max_len, return_tensors="pt").to(device)
    emb_layer = model.encoder.get_input_embeddings()
    ids = enc["input_ids"]
    x = emb_layer(ids).detach()
    special = torch.tensor(tokenizer.get_special_tokens_mask(ids[0].tolist(), already_has_special_tokens=True),
                           device=device).bool()
    base_ids = torch.where(special, ids[0], torch.full_like(ids[0], tokenizer.pad_token_id)).unsqueeze(0)
    x0 = emb_layer(base_ids).detach()

    alphas = torch.linspace(1.0 / steps, 1.0, steps, device=device).view(-1, 1, 1)
    grads_total = torch.zeros_like(x)
    chunk = 8
    for i in range(0, steps, chunk):
        al = alphas[i:i + chunk]
        xi = (x0 + al * (x - x0)).requires_grad_(True)
        mask = enc["attention_mask"].expand(xi.size(0), -1)
        tt = enc.get("token_type_ids")
        out = model(inputs_embeds=xi, attention_mask=mask,
                    token_type_ids=tt.expand(xi.size(0), -1) if tt is not None else None)
        g = torch.autograd.grad(out["score"].sum(), xi)[0]
        grads_total += g.sum(0, keepdim=True)
    attr = ((x - x0) * grads_total / steps).sum(-1)[0].detach().cpu().numpy()

    tokens = tokenizer.convert_ids_to_tokens(ids[0].tolist())
    # find the segment boundary: second run of tokens after the first separator(s)
    sep_positions = [i for i, t in enumerate(tokens) if t in (tokenizer.sep_token, tokenizer.eos_token)]
    first_sep = sep_positions[0] if sep_positions else len(tokens) // 2
    spec = special.cpu().numpy()
    seg_a = [(t, s) for i, (t, s) in enumerate(zip(tokens, attr)) if i < first_sep and not spec[i]]
    seg_b = [(t, s) for i, (t, s) in enumerate(zip(tokens, attr)) if i > first_sep and not spec[i]]
    return {
        "original": _merge_subwords([t for t, _ in seg_a], np.array([s for _, s in seg_a])),
        "simplified": _merge_subwords([t for t, _ in seg_b], np.array([s for _, s in seg_b])),
    }


def explain_pair(model, tokenizer, original: str, simplified: str, lang: str, cfg: Config,
                 domain: str = "", context: str = "", top_k: int = 5, err_thr: float = 0.5,
                 with_ig: bool = True) -> Dict:
    """Full system output for one pair: score, decision, error types, evidence and explanation."""
    row = pd.DataFrame([{"original": original, "simplified": simplified, "context": context,
                         "lang": lang, "domain": domain}])
    pred = predict(model, tokenizer, row, cfg).iloc[0]
    score = float(pred["pred_score"])
    probs = {e: float(pred[f"p_{e}"]) for e in ERROR_TYPES}
    model_errors = [e for e, p in sorted(probs.items(), key=lambda kv: -kv[1]) if p >= err_thr]
    rules = legal_diff(original, simplified, lang)

    ig = integrated_gradients(model, tokenizer, original, simplified, cfg, context) if with_ig else None
    neg_words = []
    if ig:
        neg_words = [w for w, s in sorted(ig["simplified"], key=lambda ws: ws[1])[:top_k] if s < 0]

    lines = [f"LMP score: {score:.1f}/10 -> {pred['pred_decision']}."]
    if model_errors:
        lines.append("Predicted legal error type(s): " + "; ".join(
            f"{e} (p={probs[e]:.2f}): {ERROR_DESCRIPTIONS[e]}" for e in model_errors) + ".")
    elif score < cfg.preserved_min:
        top = max(probs, key=probs.get)
        lines.append(f"No error type above threshold; most likely: {top} (p={probs[top]:.2f}).")
    if rules["findings"]:
        lines.append("Rule-based evidence: " + " ".join(rules["findings"]))
    if neg_words:
        lines.append("Words in the simplification that lowered the score most: " + ", ".join(f"'{w}'" for w in neg_words) + ".")
    return {
        "lmp_score": round(score, 2),
        "decision": pred["pred_decision"],
        "error_types": model_errors,
        "error_probs": {k: round(v, 3) for k, v in probs.items()},
        "rule_findings": rules["findings"],
        "rule_error_types": sorted(rules["error_types"]),
        "attributions": ig,
        "explanation": " ".join(lines),
    }


In [ ]:
%%writefile lmp/experiments.py
"""The five experiments required by the CSET 346 project brief.

Exp 1  Generic semantic similarity baselines vs. the JUDGEBERT-style LMP model
Exp 2  English vs. Hindi (+ cross-lingual zero-shot transfer)
Exp 3  New-domain generalisation (insurance <-> tenancy) and sentence-only vs. contextual input
Exp 4  Identical / unrelated sanity checks (with vs. without data augmentation)
Exp 5  Perturbation robustness, legal-error-type classification and qualitative error analysis (>=15 cases)
"""
import copy
import gc
import json
import os
from dataclasses import asdict, replace
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import torch

from .baselines import run_baselines
from .config import DECISIONS, ERROR_TYPES, Config, score_to_decision
from .data import augment, prepare_data, split_data
from .explain import explain_pair
from .legal_diff import legal_diff
from .metrics import (bootstrap_ci, error_type_metrics, regression_metrics, sanity_metrics,
                      summarize_seeds)
from .perturb import build_sanity_pairs, build_silver_dataset, perturbation_suite
from .train import predict, save_model, train_model

OURS = "Ours (multi-task LMP)"


def _md(df: pd.DataFrame) -> str:
    try:
        return df.to_markdown(index=False, floatfmt=".3f")
    except Exception:
        return df.to_string(index=False)


def _free(model):
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


class Runner:
    def __init__(self, cfg: Config, human_csv: Optional[str] = None, ablation_seeds: Optional[List[int]] = None,
                 split_seed: int = 42):
        self.cfg = cfg
        self.human = human_csv is not None
        self.ablation_seeds = ablation_seeds or cfg.seeds[:1]
        os.makedirs(cfg.output_dir, exist_ok=True)
        self.df = prepare_data(cfg, human_csv)
        self.train, self.val, self.test = split_data(self.df, cfg, seed=split_seed)
        # silver pairs are extra training data only when real human data is used
        self.extra_silver = None
        if self.human:
            silver = build_silver_dataset()
            held_out = set(self.val["original"]) | set(self.test["original"])
            # never let a silver variant of a val/test clause leak into training
            self.extra_silver = silver[~silver["original"].isin(held_out)].reset_index(drop=True)
            print(f"+ {len(self.extra_silver)} silver pairs used as extra training data")
        self.report: List[str] = []
        self.model = self.tokenizer = None
        print(f"train {len(self.train)} | val {len(self.val)} | test {len(self.test)} pairs "
              f"(grouped by clause, no leakage)")
        self.df.to_csv(os.path.join(cfg.output_dir, "dataset_used.csv"), index=False)

    # ------------------------------------------------------------------ core
    def fit_predict(self, train: pd.DataFrame, val: pd.DataFrame, eval_sets: Dict[str, pd.DataFrame],
                    cfg: Optional[Config] = None, seeds: Optional[List[int]] = None, keep_model=False,
                    augment_train=True, tag=""):
        cfg = cfg or self.cfg
        seeds = seeds or cfg.seeds
        tr = augment(train, cfg, self.extra_silver) if augment_train else train
        per_seed = {k: [] for k in eval_sets}
        for i, seed in enumerate(seeds):
            print(f"[{tag}] training seed {seed} on {len(tr)} pairs ...")
            model, tok, hist = train_model(tr, val, cfg, seed=seed)
            hist.to_csv(os.path.join(cfg.output_dir, f"history_{tag}_seed{seed}.csv"), index=False)
            for name, d in eval_sets.items():
                per_seed[name].append(predict(model, tok, d, cfg))
            if keep_model and i == 0:
                self.model, self.tokenizer = model, tok
            else:
                _free(model)
        out = {}
        for name, preds in per_seed.items():
            mean = sum(p.drop(columns=["pred_decision"]) for p in preds) / len(preds)
            mean["pred_decision"] = [score_to_decision(s, cfg) for s in mean["pred_score"]]
            mean.attrs["per_seed_scores"] = [p["pred_score"].to_numpy() for p in preds]
            out[name] = mean
        return out

    def _seed_table(self, gold, pred_frame, cfg=None):
        cfg = cfg or self.cfg
        return summarize_seeds([regression_metrics(gold, s, cfg) for s in pred_frame.attrs["per_seed_scores"]])

    def _log(self, title, df, note=""):
        text = f"\n## {title}\n\n{note}\n\n{_md(df)}\n"
        print(text)
        self.report.append(text)

    # ------------------------------------------------------------------ Exp 1
    def exp1_baselines_vs_model(self):
        cfg = self.cfg
        test = self.test.copy()
        self.sanity_test = build_sanity_pairs(test, seed=5)
        base = test[test["lmp"] >= 7] if self.human else test[test["perturbation"] == "faithful"]
        self.robust = perturbation_suite(base).reset_index(drop=True)

        preds = self.fit_predict(self.train, self.val,
                                 {"test": test, "sanity": self.sanity_test, "robust": self.robust},
                                 keep_model=True, tag="main")
        self.pred_test, self.pred_sanity, self.pred_robust = preds["test"], preds["sanity"], preds["robust"]
        save_model(self.model, self.tokenizer, os.path.join(cfg.output_dir, "lmp_model"))

        allpairs = pd.concat([test.assign(_set="test"), self.sanity_test.assign(_set="sanity"),
                              self.robust.assign(_set="robust")], ignore_index=True)
        b = run_baselines(allpairs, cfg)
        self.base_test = b[allpairs._set == "test"].set_index(test.index)
        self.base_sanity = b[allpairs._set == "sanity"].set_index(self.sanity_test.index)
        self.base_robust = b[allpairs._set == "robust"].set_index(self.robust.index)
        self.baseline_names = list(b.columns)

        y = test["lmp"].to_numpy()
        rows = []
        for name in self.baseline_names:
            m = regression_metrics(y, self.base_test[name], cfg)
            lo, hi = bootstrap_ci(y, self.base_test[name], cfg)
            rows.append({"system": name, **{k: round(v, 3) if isinstance(v, float) else v for k, v in m.items()},
                         "pearson_95CI": f"[{lo:.2f}, {hi:.2f}]"})
        ours = self._seed_table(y, self.pred_test)
        lo, hi = bootstrap_ci(y, self.pred_test["pred_score"], cfg)
        rows.append({"system": OURS, **ours, "pearson_95CI": f"[{lo:.2f}, {hi:.2f}] (ensemble)"})
        table = pd.DataFrame(rows)
        table.to_csv(os.path.join(cfg.output_dir, "exp1_main.csv"), index=False)
        self._log("Experiment 1 - Generic similarity vs. LMP model (test set)", table,
                  "Scores on the 1-10 LMP scale. over_% = predictions that over-estimate preservation by "
                  ">0.5 levels (dangerous direction). Ours = mean ± std over seeds.")
        self._plot_scatter(y)
        return table

    # ------------------------------------------------------------------ Exp 2
    def exp2_languages(self):
        cfg, test = self.cfg, self.test
        rows = []
        for lang in sorted(test["lang"].unique()):
            m = (test["lang"] == lang).to_numpy()
            y = test.loc[m, "lmp"].to_numpy()
            for name in self.baseline_names:
                rows.append({"setting": "joint (en+hi)", "test_lang": lang, "system": name,
                             **regression_metrics(y, self.base_test.loc[m, name], cfg)})
            rows.append({"setting": "joint (en+hi)", "test_lang": lang, "system": OURS,
                         **regression_metrics(y, self.pred_test.loc[m, "pred_score"], cfg)})
        langs = sorted(self.df["lang"].unique())
        if len(langs) >= 2:
            for src in langs:
                for tgt in langs:
                    if src == tgt:
                        continue
                    tr = self.train[self.train["lang"] == src]
                    va = self.val[self.val["lang"] == src]
                    te = test[test["lang"] == tgt]
                    if len(tr) == 0 or len(te) == 0:
                        continue
                    old = self.extra_silver
                    if old is not None:
                        self.extra_silver = old[old["lang"] == src]
                    p = self.fit_predict(tr, va if len(va) else self.val, {"t": te}, seeds=self.ablation_seeds,
                                         tag=f"xling_{src}2{tgt}")["t"]
                    self.extra_silver = old
                    rows.append({"setting": f"zero-shot {src}->{tgt}", "test_lang": tgt, "system": OURS,
                                 **regression_metrics(te["lmp"], p["pred_score"], cfg)})
        table = pd.DataFrame(rows).round(3)
        table.to_csv(os.path.join(cfg.output_dir, "exp2_languages.csv"), index=False)
        self._log("Experiment 2 - English vs. Hindi (and cross-lingual transfer)",
                  table[["setting", "test_lang", "system", "pearson", "spearman", "rmse", "mae", "over_%", "n"]])
        return table

    # ------------------------------------------------------------------ Exp 3
    def exp3_domain_and_context(self):
        cfg = self.cfg
        rows = []
        domains = sorted(self.df["domain"].unique())
        # in-domain numbers from the main model
        for d in domains:
            m = (self.test["domain"] == d).to_numpy()
            rows.append({"setting": "in-domain (joint training)", "test_domain": d, "system": OURS,
                         **regression_metrics(self.test.loc[m, "lmp"], self.pred_test.loc[m, "pred_score"], cfg)})
            best_b = max(self.baseline_names, key=lambda n: np.nan_to_num(
                regression_metrics(self.test["lmp"], self.base_test[n], cfg)["pearson"], nan=-1))
            rows.append({"setting": "in-domain", "test_domain": d, "system": best_b,
                         **regression_metrics(self.test.loc[m, "lmp"], self.base_test.loc[m, best_b], cfg)})
        # train on one domain, test on the other (all clauses of the other domain are unseen)
        if len(domains) >= 2:
            for src in domains:
                for tgt in domains:
                    if src == tgt:
                        continue
                    tr = self.train[self.train["domain"] == src]
                    va = self.val[self.val["domain"] == src]
                    te = self.df[(self.df["domain"] == tgt) & ~self.df["concept_id"].isin(tr["concept_id"])]
                    old = self.extra_silver
                    if old is not None:
                        self.extra_silver = old[old["domain"] == src]
                    p = self.fit_predict(tr, va if len(va) else self.val, {"t": te}, seeds=self.ablation_seeds,
                                         tag=f"domain_{src}2{tgt}")["t"]
                    self.extra_silver = old
                    rows.append({"setting": f"out-of-domain {src}->{tgt}", "test_domain": tgt, "system": OURS,
                                 **regression_metrics(te["lmp"], p["pred_score"], cfg)})
        dom_table = pd.DataFrame(rows).round(3)
        dom_table.to_csv(os.path.join(cfg.output_dir, "exp3a_domain.csv"), index=False)
        self._log("Experiment 3a - Legal-domain generalisation",
                  dom_table[["setting", "test_domain", "system", "pearson", "rmse", "mae", "over_%", "n"]])

        # sentence-only vs contextual input (same split, same seeds)
        ctx_cfg = replace(cfg, use_context=True)
        p_ctx = self.fit_predict(self.train, self.val, {"t": self.test}, cfg=ctx_cfg, seeds=self.ablation_seeds,
                                 tag="context")["t"]
        y = self.test["lmp"]
        sent_only = summarize_seeds([regression_metrics(y, s, cfg)
                                     for s in self.pred_test.attrs["per_seed_scores"][:len(self.ablation_seeds)]])
        ctx_table = pd.DataFrame([{"input": "sentence-only (original || simplified)", **sent_only},
                                  {"input": "contextual (context || original || simplified)",
                                   **self._seed_table(y, p_ctx, ctx_cfg)}])
        ctx_table.to_csv(os.path.join(cfg.output_dir, "exp3b_context.csv"), index=False)
        note = ("NOTE: with silver labels the context cannot change the gold score, so this only checks the "
                "pipeline. With human data, annotate a subset both with and without context.") if not self.human else ""
        self._log("Experiment 3b - Sentence-only vs. contextual input", ctx_table, note)
        return dom_table, ctx_table

    # ------------------------------------------------------------------ Exp 4
    def exp4_sanity(self):
        cfg = self.cfg
        kinds = self.sanity_test["perturbation"].to_numpy()
        rows = [{"system": n, **sanity_metrics(self.base_sanity[n], kinds)} for n in self.baseline_names]
        rows.append({"system": OURS + " + DA", **sanity_metrics(self.pred_sanity["pred_score"], kinds)})
        no_da = replace(cfg, data_augmentation=False)
        p = self.fit_predict(self.train, self.val, {"s": self.sanity_test}, cfg=no_da,
                             seeds=self.ablation_seeds, tag="noDA")["s"]
        rows.append({"system": OURS + " (no DA)", **sanity_metrics(p["pred_score"], kinds)})
        table = pd.DataFrame(rows).round(2)
        table.to_csv(os.path.join(cfg.output_dir, "exp4_sanity.csv"), index=False)
        self._log("Experiment 4 - Sanity checks (held-out clauses)", table,
                  "identical_>=99%: share of identical pairs scored >= 99% of the scale (higher is better). "
                  "unrelated_<=1%: share of unrelated same-domain legal pairs scored <= 1% (higher is better).")
        return table

    # ------------------------------------------------------------------ Exp 5
    def exp5_robustness_and_errors(self, n_cases: int = 20):
        cfg = self.cfg
        # 5a perturbation robustness: score drop relative to the faithful version
        r = self.robust.copy()
        r[OURS] = self.pred_robust["pred_score"].to_numpy()
        for n in self.baseline_names:
            r[n] = self.base_robust[n].to_numpy()
        systems = self.baseline_names + [OURS]
        mean = r.groupby("variant")[systems].mean()
        drop = (mean.loc["faithful"] - mean).drop(index="faithful")
        drop.loc["[average drop]"] = drop.mean()
        drop_table = drop.reset_index().round(2)
        drop_table.to_csv(os.path.join(cfg.output_dir, "exp5a_robustness_drop.csv"), index=False)
        self._log("Experiment 5a - Score drop caused by one legal edit (higher = more sensitive = better)",
                  drop_table, "Each row: mean(score of faithful simplification) - mean(score after the edit).")
        self._plot_robustness(drop.drop(index="[average drop]"))

        # 5b legal error-type classification: model head vs. rule-based legal diff
        t = self.test.copy()
        if t["errors"].notna().any():
            model_m = error_type_metrics(t["errors"], self.pred_test, ERROR_TYPES)
            rule_prob = pd.DataFrame(
                [{f"p_{e}": float(e in legal_diff(o, s, l)["error_types"]) for e in ERROR_TYPES}
                 for o, s, l in zip(t["original"], t["simplified"], t["lang"])], index=t.index)
            rule_m = error_type_metrics(t["errors"], rule_prob, ERROR_TYPES)
            et = model_m.merge(rule_m, on=["error_type", "support"], suffixes=("_model", "_rules")).round(3)
            et.to_csv(os.path.join(cfg.output_dir, "exp5b_error_types.csv"), index=False)
            self._log("Experiment 5b - Legal error-type classification (P/R/F1)", et)

        # 5c qualitative error analysis with explanations
        cases = self._select_cases(n_cases)
        out = []
        for _, c in cases.iterrows():
            ex = explain_pair(self.model, self.tokenizer, c["original"], c["simplified"], c["lang"], cfg,
                              domain=c["domain"], context=c.get("context", ""))
            out.append({
                "pair_id": c["pair_id"], "lang": c["lang"], "domain": c["domain"],
                "gold_errors": c.get("errors", ""), "original": c["original"], "simplified": c["simplified"],
                "gold_lmp": round(float(c["lmp"]), 2), "model_lmp": ex["lmp_score"], "model_decision": ex["decision"],
                "model_error_types": ";".join(ex["error_types"]),
                "BERTScore": round(float(self.base_test.loc[c.name, "BERTScore(XLM-R)"]), 2)
                if "BERTScore(XLM-R)" in self.base_test else None,
                "chrF": round(float(self.base_test.loc[c.name, "chrF"]), 2),
                "explanation": ex["explanation"],
            })
        qual = pd.DataFrame(out)
        qual.to_csv(os.path.join(cfg.output_dir, "exp5c_qualitative_cases.csv"), index=False)
        with open(os.path.join(cfg.output_dir, "exp5c_qualitative_cases.md"), "w", encoding="utf-8") as f:
            for i, q in qual.iterrows():
                f.write(f"### Case {i + 1} ({q['lang']}, {q['domain']}, gold errors: {q['gold_errors'] or 'none'})\n"
                        f"- **Original:** {q['original']}\n- **Simplified:** {q['simplified']}\n"
                        f"- **Gold LMP:** {q['gold_lmp']} | **Model:** {q['model_lmp']} ({q['model_decision']}) "
                        f"| **chrF:** {q['chrF']} | **BERTScore:** {q['BERTScore']}\n"
                        f"- **Explanation:** {q['explanation']}\n\n")
        self._log(f"Experiment 5c - Qualitative error analysis ({len(qual)} cases, full text in exp5c_qualitative_cases.md)",
                  qual[["pair_id", "lang", "gold_errors", "gold_lmp", "model_lmp", "chrF", "model_error_types"]])
        return drop_table, qual

    def _select_cases(self, n):
        t = self.test.copy()
        t["abs_err"] = (self.pred_test["pred_score"] - t["lmp"]).abs()
        picked = []
        # one per legal error type (legally difficult cases) ...
        if t["errors"].notna().any():
            for e in ERROR_TYPES:
                c = t[t["errors"].fillna("").str.contains(e)]
                c = c[~c.index.isin(picked)]
                if len(c):
                    picked.append(c.sort_values("abs_err", ascending=False).index[0])
        # ... a few preserved ones, then the largest model errors
        pres = t[t["lmp"] >= 7].index.tolist()[:3]
        picked += [i for i in pres if i not in picked]
        for i in t.sort_values("abs_err", ascending=False).index:
            if len(picked) >= n:
                break
            if i not in picked:
                picked.append(i)
        return t.loc[picked[:n]]

    # ------------------------------------------------------------------ plots + report
    def _plot_scatter(self, y):
        try:
            import matplotlib.pyplot as plt
            best = max(self.baseline_names, key=lambda n: np.nan_to_num(
                regression_metrics(y, self.base_test[n], self.cfg)["pearson"], nan=-1))
            fig, ax = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
            for a, (name, p) in zip(ax, [(best, self.base_test[best]), (OURS, self.pred_test["pred_score"])]):
                a.scatter(y, p, s=18, alpha=0.7)
                a.plot([1, 10], [1, 10], "k--", lw=1)
                a.set_title(name)
                a.set_xlabel("gold LMP")
                a.set_xlim(0.5, 10.5)
                a.set_ylim(0.5, 10.5)
            ax[0].set_ylabel("predicted LMP")
            fig.tight_layout()
            fig.savefig(os.path.join(self.cfg.output_dir, "fig_scatter.png"), dpi=150)
            plt.close(fig)
        except Exception as e:
            print("plot skipped:", e)

    def _plot_robustness(self, drop: pd.DataFrame):
        try:
            import matplotlib.pyplot as plt
            ax = drop.plot.bar(figsize=(11, 4.5), width=0.8)
            ax.set_ylabel("score drop (1-10 scale)")
            ax.set_title("Sensitivity to single legal edits")
            ax.axhline(0, color="k", lw=0.8)
            plt.xticks(rotation=30, ha="right")
            plt.tight_layout()
            plt.savefig(os.path.join(self.cfg.output_dir, "fig_robustness.png"), dpi=150)
            plt.close()
        except Exception as e:
            print("plot skipped:", e)

    def write_report(self):
        head = ("# Results - Explainable Multilingual Legal Meaning Preservation\n\n"
                f"Data: {'HUMAN-annotated' if self.human else 'SILVER (rule-generated, pipeline demo only)'} | "
                f"model: `{self.cfg.model_name}` | seeds: {self.cfg.seeds}\n")
        path = os.path.join(self.cfg.output_dir, "results.md")
        with open(path, "w", encoding="utf-8") as f:
            f.write(head + "".join(self.report))
        with open(os.path.join(self.cfg.output_dir, "config.json"), "w") as f:
            json.dump(asdict(self.cfg), f, indent=2)
        print("Report written to", path)

    def run_all(self):
        self.exp1_baselines_vs_model()
        self.exp2_languages()
        self.exp3_domain_and_context()
        self.exp4_sanity()
        self.exp5_robustness_and_errors()
        self.write_report()


In [ ]:
%%writefile lmp/generate.py
"""Build YOUR dataset: generate simplifications of real legal clauses with open LLMs on Colab.

Mirrors the JUDGEBERT setup (zero-shot prompt of Kew et al. 2023, temperature 1.0, top-p 0.9) but
uses open models and *several* systems, so the test set is not limited to one model's error profile.
"""
from typing import List

import pandas as pd
import torch


# The paper's prompt (Appendix A), translated; the language instruction is in capitals as in the paper.
PROMPTS = {
    "en": "Rewrite the complex sentence with simple sentence(s). Keep the meaning the same, but make it simpler. "
          "ANSWER IN ENGLISH!\n\nComplex: {input}",
    "hi": "जटिल वाक्य को एक या अधिक सरल वाक्यों में दोबारा लिखें। अर्थ वही रखें, लेकिन इसे सरल बनाएं। "
          "केवल हिंदी में उत्तर दें!\n\nजटिल वाक्य: {input}",
}

# Small instruction models that fit a free Colab T4 and handle English + Hindi reasonably.
DEFAULT_SYSTEMS = ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct"]


@torch.no_grad()
def simplify_with_llm(texts: List[str], langs: List[str], model_name: str, max_new_tokens: int = 160,
                      temperature: float = 1.0, top_p: float = 0.9, batch_size: int = 8) -> List[str]:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_name, padding_side="left")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None)
    outs = []
    for i in range(0, len(texts), batch_size):
        prompts = [tok.apply_chat_template([{"role": "user", "content": PROMPTS[l].format(input=t)}],
                                           tokenize=False, add_generation_prompt=True)
                   for t, l in zip(texts[i:i + batch_size], langs[i:i + batch_size])]
        enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature,
                             top_p=top_p, pad_token_id=tok.pad_token_id)
        for g in gen[:, enc["input_ids"].shape[1]:]:
            outs.append(tok.decode(g, skip_special_tokens=True).strip().split("\n\n")[0])
        print(f"  {model_name}: {min(i + batch_size, len(texts))}/{len(texts)}")
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return outs


def generate_pairs(sources: pd.DataFrame, systems: List[str] = None, seed: int = 0) -> pd.DataFrame:
    """sources: columns source_id, lang, domain, context, original -> one row per (clause, system)."""
    torch.manual_seed(seed)
    systems = systems or DEFAULT_SYSTEMS
    rows = []
    for sysname in systems:
        simp = simplify_with_llm(sources["original"].tolist(), sources["lang"].tolist(), sysname)
        part = sources.copy()
        part["simplified"] = simp
        part["system"] = sysname
        rows.append(part)
    return pd.concat(rows, ignore_index=True)


def make_annotation_sheet(pairs: pd.DataFrame, path: str, n_annotators: int = 3, seed: int = 0) -> pd.DataFrame:
    """Shuffled sheet for annotators. The `system` column is kept for analysis but should be hidden
    from annotators (as in the paper) - e.g. hide it in Google Sheets."""
    sheet = pairs.sample(frac=1.0, random_state=seed).reset_index(drop=True).copy()
    sheet.insert(0, "pair_id", [f"H{i:04d}" for i in range(len(sheet))])
    if "concept_id" not in sheet:
        sheet["concept_id"] = sheet["source_id"]
    for k in range(1, n_annotators + 1):
        sheet[f"annotator_{k}"] = float("nan")
    sheet["errors"] = ""
    sheet["notes"] = ""
    sheet.to_csv(path, index=False, encoding="utf-8-sig")   # utf-8-sig so Excel shows Hindi correctly
    print(f"Wrote {len(sheet)} pairs to {path}")
    return sheet


In [ ]:
%%writefile lmp/demo.py
"""Interactive Gradio demo for presentations: paste an original + simplified clause, get the full output."""
from .config import Config
from .explain import explain_pair


def launch_demo(model, tokenizer, cfg: Config, share: bool = True, launch: bool = True):
    import gradio as gr

    def run(original, simplified, lang):
        ex = explain_pair(model, tokenizer, original, simplified, lang, cfg)
        attr = ex["attributions"]["simplified"] if ex["attributions"] else []
        m = max([abs(s) for _, s in attr] + [1e-9])
        highlighted = [(w + " ", round(float(s) / m, 3)) for w, s in attr]   # in [-1, 1] for colouring
        probs = {k: v for k, v in ex["error_probs"].items()}
        return f"{ex['lmp_score']:.1f} / 10", ex["decision"], probs, highlighted, ex["explanation"]

    examples = [
        ["The Insured shall notify the Insurer in writing of any loss within 30 days of its occurrence, failing which the claim shall not be admissible.",
         "The insured may tell the insurer about any loss within 30 days.", "en"],
        ["The Tenant shall not sublet, assign or part with possession of the premises or any part thereof without the prior written consent of the Landlord.",
         "The tenant must not sublet any part of the premises without the landlord's written permission.", "en"],
        ["किरायेदार प्रत्येक कैलेंडर माह की 5 तारीख को या उससे पूर्व रु. 25,000 का मासिक किराया अदा करेगा।",
         "किरायेदार को हर महीने की 10 तारीख तक रु. 25,000 किराया देना होगा।", "hi"],
    ]
    demo = gr.Interface(
        fn=run,
        inputs=[gr.Textbox(label="Original legal text", lines=4), gr.Textbox(label="Simplified text", lines=4),
                gr.Radio(["en", "hi"], value="en", label="Language")],
        outputs=[gr.Textbox(label="LMP score"), gr.Textbox(label="Decision"),
                 gr.Label(label="Legal error-type probabilities", num_top_classes=9),
                 gr.HighlightedText(label="Word attributions in the simplification (red = lowered the score)"),
                 gr.Textbox(label="Explanation", lines=5)],
        examples=examples, title="Explainable Multilingual Legal Meaning Preservation",
        description="JUDGEBERT-style cross-encoder extended to English + Hindi with error types and explanations.")
    return demo.launch(share=share) if launch else demo


## 3. Look at the data
Parallel English-Hindi seed clauses and the controlled legal perturbations used for silver data, augmentation and robustness tests.

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 160)
from lmp.seed_data import seed_rows
from lmp.perturb import build_silver_dataset, PERTURBATIONS

seeds = pd.DataFrame(seed_rows())
seeds[['source_id','concept_id','lang','domain','context','original']].to_csv('data/sources_seed.csv', index=False, encoding='utf-8-sig')
silver = build_silver_dataset()
print(len(seeds), "clauses |", len(silver), "silver pairs")
display(silver.groupby(['lang','domain']).size().unstack())
display(silver.sample(6, random_state=1)[['lang','perturbation','simplified','lmp','errors']])

In [ ]:
# Try the perturbations on any sentence
s, lang, dom = "The tenant must pay the monthly rent of Rs. 25,000 by the 5th of every month.", "en", "tenancy"
for name, (fn, etype, sev) in PERTURBATIONS.items():
    print(f"{name:16s} -> {fn(s, lang, dom)}")

## 4. Configuration
* `HUMAN_CSV`: path to your annotated file (format: `data/annotation_template.csv`, see `ANNOTATION_GUIDE.md`).
  Upload it with the folder icon on the left, then set the path. `None` = silver demo.
* `MODEL_NAME`: `xlm-roberta-base` (default), `google/muril-base-cased` (Indian languages), `xlm-roberta-large` (A100).
* `FAST_MODE`: 1 seed / 8 epochs. Turn it off for the final paper numbers (3 seeds, mean ± std as in JUDGEBERT).

In [ ]:
from lmp.config import Config

HUMAN_CSV  = None            # e.g. "data/annotated_pairs.csv"
MODEL_NAME = "xlm-roberta-base"
FAST_MODE  = True

cfg = Config(model_name=MODEL_NAME, output_dir="outputs")
if FAST_MODE:
    cfg.seeds, cfg.epochs = [42], 8
cfg

## 5. (Optional) Build your own dataset with open LLMs
1. Put your real clauses in a CSV with columns `source_id, lang, domain, context, original`
   (start from `data/sources_seed.csv`; add clauses from real Indian insurance policy wordings and rent agreements).
2. Generate simplifications with two or more open LLMs (more realistic errors than one system).
3. Download the annotation sheet, rate it following `ANNOTATION_GUIDE.md` (e.g. in Google Sheets), upload, and set `HUMAN_CSV`.

In [ ]:
RUN_GENERATION = False   # set True to generate (about 5-10 min for 100 clauses x 2 models on a T4)
if RUN_GENERATION:
    from lmp.generate import generate_pairs, make_annotation_sheet
    sources = pd.read_csv("data/sources_seed.csv")
    pairs = generate_pairs(sources, systems=["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct"])
    sheet = make_annotation_sheet(pairs, "data/to_annotate.csv", n_annotators=3)
    display(sheet.head())
    from google.colab import files; files.download("data/to_annotate.csv")

## 6. Load data and split (grouped by clause, no leakage)

In [ ]:
from lmp.experiments import Runner
runner = Runner(cfg, human_csv=HUMAN_CSV)

## 7. Experiment 1 - generic semantic similarity vs. LMP model
Trains the multi-task model (score + decision + error types, asymmetric loss) and runs TF-IDF, chrF, BLEU, BERTScore, SBERT-Multi and LaBSE.

In [ ]:
exp1 = runner.exp1_baselines_vs_model()
from IPython.display import Image, display
display(exp1)
display(Image("outputs/fig_scatter.png"))

## 8. Experiment 2 - English vs. Hindi, and cross-lingual zero-shot transfer

In [ ]:
exp2 = runner.exp2_languages()
display(exp2)

## 9. Experiment 3 - new legal domain and contextual input

In [ ]:
exp3_domain, exp3_context = runner.exp3_domain_and_context()
display(exp3_domain); display(exp3_context)

## 10. Experiment 4 - identical / unrelated sanity checks (with vs. without augmentation)

In [ ]:
exp4 = runner.exp4_sanity()
display(exp4)

## 11. Experiment 5 - robustness to single legal edits, error types, qualitative analysis (>= 15 cases)

In [ ]:
exp5_drop, exp5_cases = runner.exp5_robustness_and_errors(n_cases=20)
display(exp5_drop)
display(Image("outputs/fig_robustness.png"))
display(exp5_cases[['pair_id','lang','gold_errors','gold_lmp','model_lmp','chrF','explanation']])

## 12. Full report

In [ ]:
runner.write_report()
from IPython.display import Markdown
display(Markdown(open("outputs/results.md", encoding="utf-8").read()))

## 13. Use the system on any pair

In [ ]:
from lmp.explain import explain_pair
out = explain_pair(runner.model, runner.tokenizer,
    original="The Insurer shall not be liable for any loss caused by flood, unless the Flood Extension has been purchased.",
    simplified="The insurer will not pay for flood damage.",
    lang="en", cfg=cfg)
print(out["explanation"])
{k: out[k] for k in ["lmp_score", "decision", "error_types", "rule_error_types"]}

In [ ]:
# Interactive demo for the presentation (public link valid for 72 h)
from lmp.demo import launch_demo
launch_demo(runner.model, runner.tokenizer, cfg)

## 14. Save results

In [ ]:
!zip -qr outputs.zip outputs -x "outputs/lmp_model/*"
from google.colab import files
files.download("outputs.zip")
# To keep the trained model too:  from google.colab import drive; drive.mount('/content/drive'); !cp -r outputs /content/drive/MyDrive/lmp_outputs